# Ekonometrik / Ekonomik Modelleme Notebook'u

**Konfigürasyon odaklı, yeniden kullanılabilir** bir ekonometrik modelleme
çatısı. Amaç: kodun büyük kısmına dokunmadan, yalnızca en üstteki `CONFIG`
bloğunu değiştirerek farklı veri setleri ve değişkenler üzerinde çok sayıda
ekonometrik modeli kurmak, test etmek, karşılaştırmak, en iyi modeli seçmek,
forecast üretmek ve tüm çıktıları düzenli bir Excel dosyasına yazmak.

**Desteklenen model aileleri:** OLS, ADL, Distributed Lag, ARDL, ECM,
VAR, VECM, SARIMAX, Ridge, Lasso, ElasticNet.

**Çalışma mantığı**
1. Kullanıcı yalnızca `CONFIG` bloğunu düzenler.
2. Ana kod hücreleri sabit kalır.
3. Her model aynı hedef değişken ve aynı forecast senaryosu üzerinden
   karşılaştırılır.
4. Hata veren model tüm süreci durdurmaz; hata `10_ERRORS_WARNINGS`
   sayfasına yazılır.
5. Sonuçlar tek bir Excel dosyasına aktarılır ve her test için kısa,
   net **Türkçe yorum** üretilir.

> Not: Notebook, `input_file` bulunamazsa otomatik olarak sentetik bir örnek
> veri seti üretir (bkz. `CONFIG["generate_sample_if_missing"]`). Böylece
> "tak-çalıştır" mantığında baştan sona sorunsuz çalışır.

## 1. Kurulum ve importlar

Gerekli paketler import edilir. Opsiyonel paketler (`arch`, `pmdarima`,
`linearmodels`) yoksa notebook çökmez; ilgili özellikler devre dışı kalır ve
kullanıcıya açık bir uyarı verilir.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan, het_white
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.stattools import coint, adfuller

from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit

import matplotlib
try:
    get_ipython()  # noqa: F821  (notebook ortamı -> inline backend kalsın)
except NameError:
    matplotlib.use("Agg")  # script/headless çalıştırmada güvenli
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

# --- Opsiyonel paketleri sessizce dene ---
OPTIONAL = {}
for _pkg in ["arch", "pmdarima", "linearmodels"]:
    try:
        __import__(_pkg)
        OPTIONAL[_pkg] = True
    except Exception:
        OPTIONAL[_pkg] = False

# --- Çekirdek zaman serisi bileşenleri (statsmodels) ---
HAVE = {}
try:
    from statsmodels.tsa.ardl import ARDL, ardl_select_order
    HAVE["ardl"] = True
except Exception:
    HAVE["ardl"] = False
try:
    from statsmodels.tsa.api import VAR
    from statsmodels.tsa.vector_ar.vecm import VECM, coint_johansen, select_coint_rank
    HAVE["var"] = True
except Exception:
    HAVE["var"] = False
try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    HAVE["sarimax"] = True
except Exception:
    HAVE["sarimax"] = False
try:
    from statsmodels.regression.rolling import RollingOLS
    HAVE["rolling"] = True
except Exception:
    HAVE["rolling"] = False

# Başlangıç uyarıları (Excel'de 10_ERRORS_WARNINGS sayfasına da düşer)
STARTUP_NOTES = []
for _p, _ok in OPTIONAL.items():
    if not _ok:
        STARTUP_NOTES.append(
            f"Opsiyonel paket '{_p}' bulunamadi; ilgili gelismis ozellikler devre disi "
            f"(notebook calismaya devam eder). Kurmak icin: pip install {_p}"
        )
for _k in ["ardl", "var", "sarimax"]:
    if not HAVE.get(_k):
        STARTUP_NOTES.append(f"statsmodels bileseni '{_k}' yuklenemedi; bu model ailesi atlanacak.")

print("Importlar tamam.")
print("Opsiyonel paketler:", OPTIONAL)
for _n in STARTUP_NOTES:
    print("UYARI:", _n)

## 2. CONFIG bloğu

**Notebook'un tek merkezi ayar noktası.** Kullanıcı normalde yalnızca burayı
değiştirir. Aşağıdaki örnek `CONFIG`, notebook'un ürettiği sentetik örnek veri
(`y`, `usdtry`, `policy_rate`) ile birebir çalışacak şekilde doldurulmuştur.
Kendi verinizi kullanmak için `input_file`, `date_col`, `target_col`, değişken
isimleri, transformlar, model spesifikasyonları ve forecast senaryolarını
güncelleyin. Değişken isimleri **hard-code edilmemiştir**; tamamı buradan yönetilir.

In [ ]:
CONFIG = {
    # --- Girdi / veri ---
    "input_file": "input.xlsx",
    "sheet_name": "Sheet1",
    "date_col": "date",
    "target_col": "y",

    "frequency": "M",          # D, W, M, Q, A
    "date_format": None,       # örn "%Y-%m-%d"; None ise otomatik

    "sample_start": None,      # tahmin örnekleminin başı (None = otomatik)
    "sample_end": None,        # tahmin örnekleminin sonu (None = otomatik)

    "forecast_start": None,    # None ise verinin bittiği yerin bir sonrası
    "forecast_end": None,      # None ise forecast_start + forecast_horizon

    "forecast_horizon": 12,    # forecast_end verilmezse kaç dönem ileri

    # --- Eksik veri politikası ---
    "missing_method": "interpolate",   # interpolate, ffill, bfill, dropna, none

    # --- Transformasyonlar ---
    "target_transform": "level",       # level, log, diff, logdiff, pct_change
    "exog_transforms": {
        "usdtry": "logdiff",
        "policy_rate": "level",
    },

    # --- Çalıştırılacak modeller ---
    "models_to_run": [
        "OLS",
        "ADL",
        "DISTRIBUTED_LAG",
        "ARDL",
        "ECM",
        "VAR",
        "VECM",
        "SARIMAX",
        "RIDGE",
        "LASSO",
        "ELASTICNET",
    ],

    # --- Model spesifikasyonları ---
    "model_specs": {
        "OLS": {
            "exog": ["usdtry", "policy_rate"],
            "lags_y": [],
            "lags_x": {},              # {var: [laglar]}; boşsa çağdaş (lag 0)
            "add_constant": True,
        },
        "ADL": {
            "exog": ["usdtry", "policy_rate"],
            "lags_y": [1],
            "lags_x": {"usdtry": [0, 1]},
            "add_constant": True,
        },
        "DISTRIBUTED_LAG": {
            "exog": ["usdtry", "policy_rate"],
            "lags_y": [],
            "lags_x": {"usdtry": [0, 1, 2], "policy_rate": [0, 1]},
            "add_constant": True,
        },
        "ARDL": {
            "exog": ["usdtry", "policy_rate"],
            "max_lag_y": 4,
            "max_lag_x": 4,
            "selection_criterion": "bic",   # aic/bic
        },
        "ECM": {
            "level_vars": ["usdtry", "policy_rate"],   # uzun dönem (seviye)
            "diff_vars": ["usdtry", "policy_rate"],    # kısa dönem (fark)
            "cointegration_method": "engle_granger",
            "lags_y": [1],
            "lags_x": {"usdtry": [0, 1]},
        },
        "VAR": {
            "vars": ["y", "usdtry", "policy_rate"],
            "maxlags": 6,
            "ic": "bic",
            "granger": True,
            "irf": False,
        },
        "VECM": {
            "vars": ["y", "usdtry", "policy_rate"],
            "deterministic": "ci",
            "k_ar_diff": 1,
            "coint_rank": 1,
        },
        "SARIMAX": {
            "order": [1, 0, 1],
            "seasonal_order": [0, 0, 0, 0],
            "exog": ["policy_rate"],
        },
        "RIDGE": {
            "exog": ["usdtry", "policy_rate"],
            "lags_y": [1],
            "lags_x": {"usdtry": [0, 1]},
            "alpha_grid": [0.01, 0.1, 1, 10, 100],
        },
        "LASSO": {
            "exog": ["usdtry", "policy_rate"],
            "lags_y": [1],
            "lags_x": {"usdtry": [0, 1]},
            "alpha_grid": [0.001, 0.01, 0.1, 1, 10],
        },
        "ELASTICNET": {
            "exog": ["usdtry", "policy_rate"],
            "lags_y": [1],
            "lags_x": {"usdtry": [0, 1]},
            "alpha_grid": [0.001, 0.01, 0.1, 1, 10],
            "l1_ratio_grid": [0.2, 0.5, 0.8],
        },
    },

    # --- Forecast senaryoları ---
    # Tipler: constant, growth, path, linear, shock. Verilmeyen değişken için
    # son gözlem sabit tutulur (uyarı verilir).
    "forecast_scenarios": {
        "baseline": {
            "usdtry": {"type": "growth", "monthly_rate": 0.02},
            "policy_rate": {"type": "constant", "value": 45.0},
            # Politika faizi (TR) manuel patika. Tarihler ay-başına hizalıdır;
            # verilmeyen aralar lineer interpolate, uçlar ffill/bfill ile dolar.
            "policy_rate_tr": {
                "type": "path",
                "values": {
                    "2026-06-01": 36.3846,   # Haz.26
                    "2026-07-01": 36.3846,   # Tem.26
                    "2026-08-01": 36.3846,   # Agu.26
                    "2026-09-01": 35.1563,   # Eyl.26
                    "2026-10-01": 35.1563,   # Eki.26
                    "2026-11-01": 35.1563,   # Kas.26
                    "2026-12-01": 31.6364,   # Ara.26
                    "2027-01-01": 31.6364,   # Oca.27
                    "2027-02-01": 31.6364,   # Sub.27
                    "2027-03-01": 29.0,      # Mar.27
                    "2027-04-01": 29.0,      # Nis.27
                    "2027-05-01": 29.0,      # May.27
                },
            },
        },
    },
    "active_scenario": "baseline",

    # --- Değerlendirme ---
    "test_size": 12,               # son N gözlem test seti
    "selection_metric": "bic",     # aic, bic, rmse, mae, mape, adjusted_r2, clean_score

    # --- Beklenen katsayı işaretleri (transform edilmiş isimlerle) ---
    "expected_signs": {
        "dln_usdtry": "+",
        "policy_rate": "-",
    },

    # --- Robust standart hata ---
    "cov_type": "HAC",             # nonrobust, HC0, HC1, HC3, HAC
    "hac_lags": 4,

    # --- clean_score ceza puanları (değiştirilebilir) ---
    "clean_score": {
        "start": 100,
        "bg": 20,          # Breusch-Godfrey p<0.05
        "bp": 15,          # Breusch-Pagan p<0.05
        "white": 15,       # White p<0.05
        "reset": 20,       # RESET p<0.05
        "vif": 15,         # max VIF > 10
        "insignificant": 10,   # katsayıların büyük kısmı anlamsız
        "sign": 10,        # işaret beklentisi karşılanmıyor
        "forecast": 10,    # forecast performansı kötü
        "vif_threshold": 10.0,
        "insignificant_share": 0.5,   # anlamsız pay bu eşiği geçerse ceza
        "mape_bad": 20.0,             # test MAPE bu değerin üzerindeyse forecast cezası
    },

    # --- Kayan pencere (rolling) OLS analizi (opsiyonel katman) ---
    # Zamanla değişen katsayıları çıkarır; ana model seçim/forecast akışını
    # etkilemez. enabled=False iken hiç çalışmaz.
    "rolling": {
        "enabled": True,
        "model": "OLS",        # hangi regresyon spec'i kullanılsın (OLS/ADL/DISTRIBUTED_LAG)
        "window": 12,          # kayan pencere genişliği (gözlem)
        "min_nobs": 12,        # pencere için gereken min gözlem (genelde = window)
        "step": 1,             # çıktı kaç dönemde bir raporlansın
        "mode": "coefficients",  # şu an "coefficients" destekleniyor
    },

    # --- Grafik / çıktı ---
    "make_plots": True,
    "plot_dir": "plots",
    "output_excel": "model_outputs.xlsx",

    # --- Kolaylık ---
    "generate_sample_if_missing": True,
    "random_seed": 42,
}

## 3. Veri okuma ve hazırlama

- Excel okunur, tarih kolonu `datetime`'a çevrilir ve index yapılır.
- Frekansa göre sıralanır; eksik tarihler ve eksik değerler raporlanır.
- Eksik değerler `CONFIG["missing_method"]` ile (interpolate / ffill / bfill /
  dropna / none) işlenir.
- Transformasyonlar uygulanır: `level`, `log`, `diff`, `logdiff`, `pct_change`.
- Transform edilmiş değişken isimleri sistematik üretilir: `ln_x`, `d_x`,
  `dln_x`, `pct_x`.
- Log/logdiff yapılacak değişkende sıfır/negatif değer varsa **hata değil,
  uyarı** verilir ve o transform `level` olarak alınır.

In [ ]:
# Frekans kodu eşlemesi (pandas 3.x uyumlu). Ay/çeyrek/yıl için hem dönem-sonu
# (ME/QE/YE) hem dönem-başı (MS/QS/YS) kodları desteklenir.
FREQ_MAP = {"D": "D", "W": "W",
            "M": "ME", "ME": "ME", "MS": "MS",
            "Q": "QE", "QE": "QE", "QS": "QS",
            "A": "YE", "Y": "YE", "YE": "YE", "YS": "YS", "AS": "YS"}


def resolve_frequency(index, freq_code):
    """Veri index'ine göre uygun pandas frekansını (dönem-başı/sonu dahil) belirler.

    Bu, ay-başı (2014-01-01) veriyi ay-sonu (ME) takvimine sabitleyip tüm değerleri
    NaN'a düşüren anchor uyuşmazlığını önler. Önce frekans veriden çıkarılmaya
    çalışılır; olmazsa kısayol kodu (M/Q/A) verinin ilk gününe göre başa/sona
    hizalanır.
    """
    # 1) Frekansı doğrudan veriden çıkarmayı dene (en güvenilir yol)
    try:
        inferred = pd.infer_freq(index)
    except Exception:
        inferred = None
    if inferred:
        return inferred
    # 2) Kısayol kodunu anchor-farkındalı çöz
    code = str(freq_code).upper()
    try:
        first_day = int(index.min().day)
        first_month = int(index.min().month)
    except Exception:
        first_day, first_month = 1, 1
    if code in ("M", "ME", "MS"):
        return "MS" if first_day == 1 else "ME"
    if code in ("Q", "QE", "QS"):
        return "QS" if first_day == 1 else "QE"
    if code in ("A", "Y", "YE", "YS", "AS"):
        return "YS" if (first_day == 1 and first_month == 1) else "YE"
    return FREQ_MAP.get(code, freq_code)

# Transform ön ekleri
_TR_PREFIX = {"level": "", "log": "ln_", "diff": "d_", "logdiff": "dln_", "pct_change": "pct_"}


def transformed_name(col, transform):
    """Bir değişken ve transform için sistematik kolon adı üretir."""
    if transform == "level":
        return col
    return _TR_PREFIX.get(transform, transform + "_") + col


def apply_transform(series, transform):
    """Bir seriye transform uygular. Log/logdiff'te pozitiflik gerekir."""
    s = pd.Series(series).astype(float)
    if transform == "level":
        return s
    if transform == "log":
        return np.log(s)
    if transform == "diff":
        return s.diff()
    if transform == "logdiff":
        return np.log(s).diff()
    if transform == "pct_change":
        return s.pct_change()
    raise ValueError(f"Bilinmeyen transform: {transform}")


def invert_transform_step(prev_level, modeled_value, transform):
    """Bir dönemlik modellenmiş değeri (transform edilmiş) seviyeye çevirir."""
    if transform == "level":
        return modeled_value
    if transform == "log":
        return np.exp(modeled_value)
    if transform == "diff":
        return prev_level + modeled_value
    if transform == "logdiff":
        return prev_level * np.exp(modeled_value)
    if transform == "pct_change":
        return prev_level * (1.0 + modeled_value)
    raise ValueError(f"Bilinmeyen transform: {transform}")


def generate_sample_data(config):
    """input_file yoksa kullanılan sentetik örnek veri (aylık, 120 gözlem)."""
    seed = config.get("random_seed", 42)
    rng = np.random.default_rng(seed)
    n = 120
    dates = pd.date_range("2016-01-31", periods=n, freq="ME")

    # Politika faizi: sınırlandırılmış rastgele yürüyüş
    pr = np.zeros(n); pr[0] = 8.0
    for t in range(1, n):
        pr[t] = np.clip(pr[t - 1] + rng.normal(0, 1.2), 5, 50)

    # Kur: pozitif, birikimli büyüme
    growth = rng.normal(0.015, 0.02, n)
    usd = 3.0 * np.cumprod(1 + growth)

    dln_usd = np.concatenate([[0.0], np.diff(np.log(usd))])
    # Hedef (ör. bir endeks): AR(1) + kur şoku (+) + faiz (-)
    y = np.zeros(n); y[0] = 100.0
    for t in range(1, n):
        y[t] = (17.0 + 0.85 * y[t - 1] + 40.0 * dln_usd[t]
                - 0.15 * pr[t] + rng.normal(0, 1.0))

    return pd.DataFrame({
        config["date_col"]: dates,
        config["target_col"]: y,
        "usdtry": usd,
        "policy_rate": pr,
    })


def load_data(config):
    """Excel/veri okur; yoksa (ve izin varsa) sentetik örnek veri üretir."""
    path = Path(config["input_file"])
    if path.exists():
        raw = pd.read_excel(path, sheet_name=config.get("sheet_name", 0))
    else:
        if config.get("generate_sample_if_missing", True):
            print(f"'{path}' bulunamadi -> sentetik ornek veri uretiliyor.")
            raw = generate_sample_data(config)
        else:
            raise FileNotFoundError(
                f"Girdi dosyasi bulunamadi: {path}. CONFIG['input_file'] kontrol edin "
                f"veya generate_sample_if_missing=True yapin."
            )
    return raw


def prepare_data(raw_df, config):
    """Ham veriyi temizler, transform eder ve modellemeye hazır hale getirir.

    Meta bilgileri (registry, hedef kolonlar, uyarılar) prepared_df.attrs['meta']
    içinde saklanır; böylece istenen fonksiyon imzaları korunur.
    """
    warns = []
    df = raw_df.copy()
    dcol = config["date_col"]
    if dcol not in df.columns:
        raise KeyError(f"Tarih kolonu '{dcol}' veride bulunamadi. Mevcut kolonlar: {list(df.columns)}")

    df[dcol] = pd.to_datetime(df[dcol], format=config.get("date_format"), errors="coerce")
    if df[dcol].isna().any():
        warns.append("Bazi tarihler ayristirilamadi (NaT); ilgili satirlar dusuruldu.")
        df = df.dropna(subset=[dcol])
    if df[dcol].duplicated().any():
        warns.append("Tarih kolonunda tekrar eden gozlemler var; ilk kayitlar tutuldu.")
        df = df.drop_duplicates(subset=[dcol], keep="first")

    df = df.sort_values(dcol).set_index(dcol)
    # Frekansı veriye göre anchor-farkındalı çöz (ay-başı/ay-sonu vb.)
    freq = resolve_frequency(df.index, config["frequency"])

    # Tam takvim ile eksik tarih tespiti (+ güvenlik: takvim veriyle örtüşmezse
    # yeniden örneklemeden orijinal index korunur ki veri NaN'a düşmesin)
    try:
        full_idx = pd.date_range(df.index.min(), df.index.max(), freq=freq)
        overlap = df.index.intersection(full_idx)
        if len(df.index) > 0 and len(overlap) < 0.5 * len(df.index):
            missing_dates = pd.DatetimeIndex([])
            warns.append(
                f"Frekans takvimi ('{freq}') veriyle yeterince ortusmedi "
                f"(ortusen: {len(overlap)}/{len(df.index)}); yeniden orneklemeden "
                f"orijinal tarih index'i korundu."
            )
        else:
            missing_dates = full_idx.difference(df.index)
            if len(missing_dates) > 0:
                warns.append(f"{len(missing_dates)} eksik tarih tespit edildi ve takvime eklendi.")
            df = df.reindex(full_idx)
            df.index.name = dcol
    except Exception as e:
        missing_dates = pd.DatetimeIndex([])
        warns.append(f"Frekans yeniden orneklemesi yapilamadi ({e}); veri oldugu gibi kullanildi.")

    # Eksik değer raporu
    missing_report = df.isna().sum()
    total_missing = int(missing_report.sum())
    if total_missing > 0:
        warns.append(f"Toplam {total_missing} eksik deger var; '{config.get('missing_method')}' ile islendi.")

    method = config.get("missing_method", "interpolate")
    if method == "interpolate":
        df = df.interpolate(method="time").ffill().bfill()
    elif method == "ffill":
        df = df.ffill()
    elif method == "bfill":
        df = df.bfill()
    elif method == "dropna":
        df = df.dropna()
    # "none" -> dokunma

    # --- Transformlar ---
    registry = {}
    base_to_modeled = {}

    def add_transform(base, tr):
        if base not in df.columns:
            warns.append(f"Degisken '{base}' veride yok; transform atlandi.")
            return None
        if tr in ("log", "logdiff") and (df[base].dropna() <= 0).any():
            warns.append(f"'{base}' sifir/negatif deger iceriyor; '{tr}' yerine 'level' kullanildi.")
            registry[base] = {"base": base, "transform": "level"}
            base_to_modeled[base] = base
            return base
        try:
            name = transformed_name(base, tr)
            df[name] = apply_transform(df[base], tr)
        except Exception as e:
            warns.append(f"'{base}' icin '{tr}' transformu basarisiz ({e}); level kullanildi.")
            registry[base] = {"base": base, "transform": "level"}
            base_to_modeled[base] = base
            return base
        registry[name] = {"base": base, "transform": tr}
        base_to_modeled[base] = name
        return name

    target_base = config["target_col"]
    target_tr = config.get("target_transform", "level")
    target_modeled = add_transform(target_base, target_tr)
    for base, tr in config.get("exog_transforms", {}).items():
        add_transform(base, tr)

    # Örneklem kırpma
    ss, se = config.get("sample_start"), config.get("sample_end")
    if ss is not None:
        df = df[df.index >= pd.to_datetime(ss)]
    if se is not None:
        df = df[df.index <= pd.to_datetime(se)]

    df.attrs["meta"] = {
        "registry": registry,
        "base_to_modeled": base_to_modeled,
        "target_base": target_base,
        "target_modeled": target_modeled,
        "target_transform": registry.get(target_modeled, {}).get("transform", target_tr),
        "freq": freq,
        "warnings": warns,
        "missing_report": missing_report,
        "missing_dates": missing_dates,
    }
    return df

## 4. Yardımcı fonksiyonlar

Lag üretimi, regresyon matrisi kurulumu, güvenli model fit'i, metrik ve
**Türkçe yorum** üreten fonksiyonlar. Bu hücre modellerin ortak altyapısıdır.

In [ ]:
from statsmodels.stats.diagnostic import acorr_breusch_godfrey, linear_reset

# --- Metrikler (NaN ve sıfır-gerçek-değere karşı güvenli) ---
def _rmse(actual, fitted):
    a, f = np.asarray(actual, float), np.asarray(fitted, float)
    m = ~(np.isnan(a) | np.isnan(f))
    if m.sum() == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[m] - f[m]) ** 2)))


def _mae(actual, fitted):
    a, f = np.asarray(actual, float), np.asarray(fitted, float)
    m = ~(np.isnan(a) | np.isnan(f))
    if m.sum() == 0:
        return np.nan
    return float(np.mean(np.abs(a[m] - f[m])))


def _mape(actual, fitted):
    """MAPE; gerçek değeri sıfır olan gözlemler atlanır (hata vermez)."""
    a, f = np.asarray(actual, float), np.asarray(fitted, float)
    m = ~(np.isnan(a) | np.isnan(f)) & (a != 0)
    if m.sum() == 0:
        return np.nan
    return float(np.mean(np.abs((a[m] - f[m]) / a[m])) * 100.0)


def make_lags(df, col, lags):
    """Verilen değişken için lag kolonları üretir: L1_col, L2_col, ..."""
    out = pd.DataFrame(index=df.index)
    for L in lags:
        out[f"L{L}_{col}"] = df[col].shift(L)
    return out


def build_regression_matrix(prepared_df, spec, config):
    """Model spesifikasyonuna göre y, X ve 'terim' meta bilgisini üretir.

    Dönen `terms` listesi forecast aşamasında X satırını yeniden kurmak için
    kullanılır. Her terim: {name, tcol (transform edilmiş kolon), base (orijinal
    değişken), lag, is_y}.
    """
    meta = prepared_df.attrs["meta"]
    b2m = meta["base_to_modeled"]
    target_modeled = meta["target_modeled"]

    terms = []
    # Hedef değişken lagleri
    for L in spec.get("lags_y", []):
        terms.append({"name": f"L{L}_{target_modeled}", "tcol": target_modeled,
                      "base": meta["target_base"], "lag": int(L), "is_y": True})
    # Bağımsız değişkenler (ve lagleri)
    for base in spec.get("exog", []):
        tcol = b2m.get(base, base)
        if tcol not in prepared_df.columns:
            raise KeyError(f"Bagimsiz degisken '{base}' (kolon '{tcol}') veride bulunamadi.")
        lags = spec.get("lags_x", {}).get(base, [0])
        for L in lags:
            name = tcol if L == 0 else f"L{L}_{tcol}"
            terms.append({"name": name, "tcol": tcol, "base": base, "lag": int(L), "is_y": False})

    # X matrisini kur
    X = pd.DataFrame(index=prepared_df.index)
    for t in terms:
        X[t["name"]] = prepared_df[t["tcol"]].shift(t["lag"])
    y = prepared_df[target_modeled].copy()

    add_constant = bool(spec.get("add_constant", True))
    return {"y": y, "X": X, "terms": terms, "add_constant": add_constant,
            "target_modeled": target_modeled}


def align_xy(y, X):
    """y ve X'i ortak (NaN'sız) gözlemlere hizalar."""
    data = pd.concat([y.rename("__y__"), X], axis=1).dropna()
    return data["__y__"], data.drop(columns="__y__")


# --- p-value yorumu ---
def interpret_pvalue(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "hesaplanamadi"
    if p < 0.01:
        return "guclu anlamli"
    if p < 0.05:
        return "istatistiksel olarak anlamli"
    if p < 0.10:
        return "zayif anlamli"
    return "anlamli degil"


# --- Beklenen işaret kontrolü ---
def _strip_lag_prefix(name):
    import re
    return re.sub(r"^L\d+_", "", str(name))


def expected_sign_for(var_name, expected_signs):
    base = _strip_lag_prefix(var_name)
    if base in expected_signs:
        return expected_signs[base]
    if var_name in expected_signs:
        return expected_signs[var_name]
    return None


def check_expected_signs(coef_df, expected_signs):
    """Katsayı işaretlerinin beklentiyle uyumunu kontrol eder."""
    total, ok = 0, 0
    signs_ok = []
    for _, row in coef_df.iterrows():
        var = row["variable"]
        exp = expected_sign_for(var, expected_signs)
        if exp is None or var == "const" or pd.isna(row.get("coef")):
            signs_ok.append(None)
            continue
        total += 1
        good = (row["coef"] > 0 and exp == "+") or (row["coef"] < 0 and exp == "-")
        signs_ok.append(bool(good))
        ok += int(good)
    if total == 0:
        return None, np.nan, signs_ok
    return (ok == total), ok / total, signs_ok


def significant_coef_ratio(coef_df, alpha=0.05):
    sub = coef_df[coef_df["variable"] != "const"]
    p = pd.to_numeric(sub["p_value"], errors="coerce").dropna()
    if len(p) == 0:
        return np.nan
    return float((p < alpha).mean())

### 4.1 Tanı testleri ve VIF

Otokorelasyon (Breusch-Godfrey, Ljung-Box, Durbin-Watson), değişen varyans
(Breusch-Pagan, White), normallik (Jarque-Bera), spesifikasyon (Ramsey RESET)
ve çoklu doğrusal bağlantı (VIF). Küçük örneklemde bazı testler çalışmayabilir;
bu durumda `NaN` döner ama süreç durmaz.

In [ ]:
def full_diagnostics(res, config):
    """statsmodels OLS results nesnesinden tam tanı testi seti (p-value'lar)."""
    d = {k: np.nan for k in ["bg_pvalue", "ljungbox_pvalue", "bp_pvalue",
                             "white_pvalue", "jb_pvalue", "reset_pvalue", "durbin_watson"]}
    resid = np.asarray(res.resid)
    n = int(res.nobs)
    hac = int(config.get("hac_lags", 4)) or 4
    try:
        bg = acorr_breusch_godfrey(res, nlags=min(hac, max(1, n // 4)))
        d["bg_pvalue"] = float(bg[1])
    except Exception:
        pass
    try:
        lags = min(10, max(1, n // 5))
        lb = acorr_ljungbox(resid, lags=[lags], return_df=True)
        d["ljungbox_pvalue"] = float(lb["lb_pvalue"].iloc[-1])
    except Exception:
        pass
    try:
        bp = het_breuschpagan(resid, res.model.exog)
        d["bp_pvalue"] = float(bp[1])
    except Exception:
        pass
    try:
        wh = het_white(resid, res.model.exog)
        d["white_pvalue"] = float(wh[1])
    except Exception:
        pass
    try:
        jb = jarque_bera(resid)
        d["jb_pvalue"] = float(jb[1])
    except Exception:
        pass
    try:
        d["durbin_watson"] = float(durbin_watson(resid))
    except Exception:
        pass
    try:
        rs = linear_reset(res, power=2, use_f=True)
        d["reset_pvalue"] = float(rs.pvalue)
    except Exception:
        pass
    return d


def residual_only_diagnostics(resid):
    """Sadece residual gerektiren testler (TS modelleri: SARIMAX/VAR/VECM)."""
    d = {k: np.nan for k in ["bg_pvalue", "ljungbox_pvalue", "bp_pvalue",
                             "white_pvalue", "jb_pvalue", "reset_pvalue", "durbin_watson"]}
    resid = np.asarray(pd.Series(resid).dropna())
    n = len(resid)
    if n < 5:
        return d
    try:
        lags = min(10, max(1, n // 5))
        lb = acorr_ljungbox(resid, lags=[lags], return_df=True)
        d["ljungbox_pvalue"] = float(lb["lb_pvalue"].iloc[-1])
    except Exception:
        pass
    try:
        jb = jarque_bera(resid)
        d["jb_pvalue"] = float(jb[1])
    except Exception:
        pass
    try:
        d["durbin_watson"] = float(durbin_watson(resid))
    except Exception:
        pass
    return d


def compute_vif(X):
    """Her değişken için VIF tablosu üretir (const hariç)."""
    if X is None or X.shape[1] == 0:
        return pd.DataFrame(columns=["variable", "VIF"])
    Xc = sm.add_constant(X, has_constant="add")
    rows = []
    cols = list(Xc.columns)
    for i, col in enumerate(cols):
        if col == "const":
            continue
        try:
            v = float(variance_inflation_factor(Xc.values, i))
        except Exception:
            v = np.nan
        rows.append({"variable": col, "VIF": v})
    return pd.DataFrame(rows)


def interpret_diagnostics(diag):
    """Tanı testlerini kısa Türkçe cümlelere çevirir."""
    c = {}
    bg = diag.get("bg_pvalue")
    c["bg"] = ("otokorelasyon bulgusu yok" if (bg is not None and not np.isnan(bg) and bg >= 0.05)
               else ("otokorelasyon riski var" if (bg is not None and not np.isnan(bg)) else "BG hesaplanamadi"))
    lb = diag.get("ljungbox_pvalue")
    c["ljungbox"] = ("Ljung-Box: artiklarda otokorelasyon yok" if (lb is not None and not np.isnan(lb) and lb >= 0.05)
                     else ("Ljung-Box: artiklarda otokorelasyon var" if (lb is not None and not np.isnan(lb)) else "Ljung-Box hesaplanamadi"))
    bp = diag.get("bp_pvalue")
    c["bp"] = ("degisen varyans bulgusu yok" if (bp is not None and not np.isnan(bp) and bp >= 0.05)
               else ("degisen varyans riski var" if (bp is not None and not np.isnan(bp)) else "BP hesaplanamadi"))
    wh = diag.get("white_pvalue")
    c["white"] = ("White: sabit varyans reddedilemiyor" if (wh is not None and not np.isnan(wh) and wh >= 0.05)
                  else ("White: degisen varyans riski var" if (wh is not None and not np.isnan(wh)) else "White hesaplanamadi"))
    jb = diag.get("jb_pvalue")
    c["jb"] = ("residual normal dagiliyor" if (jb is not None and not np.isnan(jb) and jb >= 0.05)
               else ("residual normal dagilmiyor" if (jb is not None and not np.isnan(jb)) else "JB hesaplanamadi"))
    rs = diag.get("reset_pvalue")
    c["reset"] = ("model spesifikasyonu uygun gorunuyor" if (rs is not None and not np.isnan(rs) and rs >= 0.05)
                  else ("model spesifikasyonu sorunlu olabilir" if (rs is not None and not np.isnan(rs)) else "RESET hesaplanamadi"))
    dw = diag.get("durbin_watson")
    if dw is not None and not np.isnan(dw):
        if dw < 1.5:
            c["dw"] = f"Durbin-Watson={dw:.2f}: pozitif otokorelasyon isareti"
        elif dw > 2.5:
            c["dw"] = f"Durbin-Watson={dw:.2f}: negatif otokorelasyon isareti"
        else:
            c["dw"] = f"Durbin-Watson={dw:.2f}: otokorelasyon isareti zayif"
    else:
        c["dw"] = "Durbin-Watson hesaplanamadi"
    return c


def interpret_vif(max_vif, threshold=10.0):
    if max_vif is None or np.isnan(max_vif):
        return "VIF hesaplanamadi"
    if max_vif > threshold:
        return f"coklu dogrusal baglanti riski yuksek (max VIF={max_vif:.1f})"
    return f"coklu dogrusal baglanti dusuk (max VIF={max_vif:.1f})"


def calculate_clean_score(diag, config, expected_ok, insignificant_share, mape_test, max_vif):
    """Modelin genel sağlığını 0-100 arası tek skora indirger."""
    cs = config["clean_score"]
    score = float(cs["start"])
    reasons = []

    def bad(key):
        v = diag.get(key)
        return (v is not None) and (not np.isnan(v)) and (v < 0.05)

    if bad("bg_pvalue"):
        score -= cs["bg"]; reasons.append("otokorelasyon (BG)")
    if bad("bp_pvalue"):
        score -= cs["bp"]; reasons.append("degisen varyans (BP)")
    if bad("white_pvalue"):
        score -= cs["white"]; reasons.append("degisen varyans (White)")
    if bad("reset_pvalue"):
        score -= cs["reset"]; reasons.append("spesifikasyon (RESET)")
    if max_vif is not None and not np.isnan(max_vif) and max_vif > cs["vif_threshold"]:
        score -= cs["vif"]; reasons.append("coklu dogrusal baglanti (VIF)")
    if insignificant_share is not None and not np.isnan(insignificant_share) and insignificant_share > cs["insignificant_share"]:
        score -= cs["insignificant"]; reasons.append("katsayilarin cogu anlamsiz")
    if expected_ok is False:
        score -= cs["sign"]; reasons.append("isaret beklentisi karsilanmiyor")
    if mape_test is not None and not np.isnan(mape_test) and mape_test > cs["mape_bad"]:
        score -= cs["forecast"]; reasons.append("forecast performansi zayif")

    return max(0.0, score), reasons


def make_failed_result(model_name, model_type, error):
    """Standart 'failed' sonuç sözlüğü."""
    return {
        "model_name": model_name, "model_type": model_type, "status": "failed",
        "fit_object": None, "coefficients": pd.DataFrame(),
        "diagnostics": {}, "vif": pd.DataFrame(), "metrics": {},
        "forecast": pd.DataFrame(), "comments": {}, "error": str(error), "_meta": {},
    }


def safe_fit_model(fit_fn, model_name, model_type):
    """Model fit'ini güvenli çalıştırır; hata olursa 'failed' sonuç döner."""
    try:
        return fit_fn()
    except Exception as e:
        import traceback
        return make_failed_result(model_name, model_type,
                                  f"{type(e).__name__}: {e} | {traceback.format_exc().splitlines()[-1]}")

## 5. Model aileleri

Her model **standart bir sonuç sözlüğü** döndürür:
`{model_name, model_type, status, fit_object, coefficients, diagnostics, vif,
metrics, forecast, comments, error, _meta}`. `_meta`, forecast aşamasının
ihtiyaç duyduğu tahmin fonksiyonu ve terim yapısını taşır.

Tüm regresyon tabanlı modeller (OLS/ADL/DL/ARDL/Ridge/Lasso/ElasticNet) ortak
bir **recursive (özyinelemeli) forecast** altyapısını paylaşır; bu altyapı hem
test seti backtest'i hem de gerçek forecast için kullanılır. ECM, SARIMAX, VAR
ve VECM kendi tahmin mantıklarını kullanır.

In [ ]:
# ---------- Ortak: seviye geri-dönüşümü ve recursive forecaster ----------
def invert_transform_array(prev_level, modeled, transform):
    prev_level = np.asarray(prev_level, float)
    modeled = np.asarray(modeled, float)
    if transform == "level":
        return modeled
    if transform == "log":
        return np.exp(modeled)
    if transform == "diff":
        return prev_level + modeled
    if transform == "logdiff":
        return prev_level * np.exp(modeled)
    if transform == "pct_change":
        return prev_level * (1.0 + modeled)
    return modeled


def make_linreg_predictor(params, add_constant):
    def predict(fvals):
        yhat = params.get("const", 0.0) if add_constant else 0.0
        for name, val in fvals.items():
            yhat += params.get(name, 0.0) * val
        return float(yhat)
    return predict


def make_sklearn_predictor(pipe, feature_order):
    def predict(fvals):
        x = np.array([[fvals[c] for c in feature_order]], float)
        return float(pipe.predict(x)[0])
    return predict


def recursive_linreg_forecast(meta_lr, exog_series, mod_init, lvl_init, n_hist):
    """Lagli regresyon modelleri için özyinelemeli forecast (seviye + modellenmiş)."""
    terms = meta_lr["terms"]
    tr = meta_lr["target_transform"]
    predict = meta_lr["predict_scalar"]
    N = len(mod_init)
    mod = np.array(mod_init, float)
    lvl = np.array(lvl_init, float)
    for i in range(n_hist, N):
        fvals, ok = {}, True
        for t in terms:
            src = mod if t["is_y"] else exog_series.get(t["tcol"])
            if src is None:
                ok = False; break
            j = i - t["lag"]
            if j < 0 or j >= N or np.isnan(src[j]):
                ok = False; break
            fvals[t["name"]] = src[j]
        if not ok:
            mod[i] = np.nan; lvl[i] = np.nan; continue
        yhat = predict(fvals)
        mod[i] = yhat
        prev = lvl[i - 1] if i - 1 >= 0 else np.nan
        lvl[i] = float(invert_transform_step(prev, yhat, tr))
    return mod, lvl


def _cov_kwargs(config):
    ct = config.get("cov_type", "nonrobust")
    if ct in (None, "nonrobust"):
        return {}
    if ct == "HAC":
        return {"cov_type": "HAC", "cov_kwds": {"maxlags": int(config.get("hac_lags", 4))}}
    return {"cov_type": ct}


def coef_table_from_sm(res, expected_signs):
    """statsmodels sonuç nesnesinden standart katsayı tablosu."""
    params = res.params
    try:
        ci = res.conf_int()
        ci.columns = [0, 1]
    except Exception:
        ci = None
    rows = []
    for name in params.index:
        try:
            p = float(res.pvalues[name])
        except Exception:
            p = np.nan
        try:
            se = float(res.bse[name])
        except Exception:
            se = np.nan
        try:
            tv = float(res.tvalues[name])
        except Exception:
            tv = np.nan
        lo = float(ci.loc[name, 0]) if ci is not None else np.nan
        hi = float(ci.loc[name, 1]) if ci is not None else np.nan
        rows.append({"variable": str(name), "coef": float(params[name]), "std_error": se,
                     "t_stat": tv, "p_value": p, "conf_low": lo, "conf_high": hi,
                     "significance_comment": interpret_pvalue(p)})
    df = pd.DataFrame(rows)
    _, _, signs_ok = check_expected_signs(df, expected_signs)
    df["expected_sign"] = [expected_sign_for(v, expected_signs) for v in df["variable"]]
    df["sign_ok"] = signs_ok
    return df


def _level_train_metrics(fitted_mod, y_index, prepared_df, tr, target_base):
    prev = prepared_df[target_base].shift(1).reindex(y_index).values
    actual = prepared_df[target_base].reindex(y_index).values
    lvlfit = invert_transform_array(prev, np.asarray(fitted_mod, float), tr)
    return _rmse(actual, lvlfit), _mae(actual, lvlfit), _mape(actual, lvlfit)


def _gaussian_ic(rss, n, k):
    """statsmodels OLS ile birebir aynı Gauss olabilirlik AIC/BIC (aynı ölçek).

    Böylece sklearn/VAR/VECM gibi native IC'si farklı ölçekte olan modeller,
    OLS/SARIMAX ile karşılaştırılabilir hale gelir.
    """
    if rss is None or n is None or rss <= 0 or n <= 0:
        return np.nan, np.nan
    base = n * np.log(rss / n) + n * (1.0 + np.log(2.0 * np.pi))
    return base + 2 * k, base + k * np.log(n)


def _metrics_dict(r2, adj, aic, bic, llf, nobs, tr3, te3, max_vif, sig_ratio, exp_ok, clean):
    return {
        "r2": r2, "adj_r2": adj, "aic": aic, "bic": bic, "loglik": llf, "nobs": nobs,
        "rmse_train": tr3[0], "mae_train": tr3[1], "mape_train": tr3[2],
        "rmse_test": te3[0], "mae_test": te3[1], "mape_test": te3[2],
        "max_vif": max_vif, "significant_coef_ratio": sig_ratio,
        "expected_signs_ok": exp_ok, "clean_score": clean,
    }


def build_comments(diag, vif, config, model_type, extra=None):
    dc = interpret_diagnostics(diag)
    max_vif = float(vif["VIF"].max()) if len(vif) > 0 else np.nan
    dc["vif"] = interpret_vif(max_vif, config["clean_score"]["vif_threshold"])
    if extra:
        dc.update(extra)
    return dc


def _make_result(model_name, model_type, res_obj, coef_df, diag, vif, metrics, comments, meta_obj):
    return {"model_name": model_name, "model_type": model_type, "status": "success",
            "fit_object": res_obj, "coefficients": coef_df, "diagnostics": diag, "vif": vif,
            "metrics": metrics, "forecast": pd.DataFrame(), "comments": comments,
            "error": None, "_meta": meta_obj}

### 5.1–5.4 OLS / ADL / Distributed Lag / (ARDL çekirdeği)

Bu modeller tek bir ortak `_fit_regression` fonksiyonuyla kurulur. Fark
yalnızca `CONFIG` içindeki lag yapısıdır. `cov_type` ile robust (HC0/HC1/HC3/
HAC) standart hatalar desteklenir.

In [ ]:
def _pipeline(mode, alpha, l1):
    if "ridge" in mode:
        est = Ridge(alpha=alpha)
    elif "lasso" in mode:
        est = Lasso(alpha=alpha, max_iter=100000)
    else:
        est = ElasticNet(alpha=alpha, l1_ratio=(l1 if l1 else 0.5), max_iter=100000)
    return Pipeline([("scaler", StandardScaler()), ("model", est)])


def _cv_fit_sklearn(mode, X, y, config, spec):
    alphas = spec.get("alpha_grid", [0.01, 0.1, 1, 10])
    l1s = spec.get("l1_ratio_grid", [0.5]) if mode.endswith("en") else [None]
    n = len(y)
    test_size = config.get("test_size", 12) or 12
    nsplit = int(min(5, max(2, n // max(1, test_size))))
    tscv = TimeSeriesSplit(n_splits=nsplit)
    best = None
    for a in alphas:
        for l1 in l1s:
            errs = []
            for tr_i, te_i in tscv.split(X):
                pipe = _pipeline(mode, a, l1)
                pipe.fit(X[tr_i], y[tr_i])
                errs.append(_rmse(y[te_i], pipe.predict(X[te_i])))
            m = np.nanmean(errs)
            if best is None or m < best[0]:
                best = (m, a, l1)
    _, a, l1 = best
    pipe = _pipeline(mode, a, l1)
    pipe.fit(X, y)
    pipe._best_alpha, pipe._best_l1 = a, l1
    return pipe


def coef_table_sklearn(pipe, feat_cols, add_c, expected_signs):
    scaler = pipe.named_steps["scaler"]
    est = pipe.named_steps["model"]
    scale = np.where(scaler.scale_ == 0, 1.0, scaler.scale_)
    coef_scaled = np.ravel(est.coef_)
    coef_orig = coef_scaled / scale
    rows = []
    if add_c:
        const = float(est.intercept_) - float(np.sum(coef_scaled * scaler.mean_ / scale))
        rows.append({"variable": "const", "coef": const, "std_error": np.nan, "t_stat": np.nan,
                     "p_value": np.nan, "conf_low": np.nan, "conf_high": np.nan,
                     "significance_comment": "sabit terim"})
    for c, co in zip(feat_cols, coef_orig):
        comment = "sifira dusuruldu" if abs(co) < 1e-10 else "duzenlilestirilmis katsayi (p-value yok)"
        rows.append({"variable": c, "coef": float(co), "std_error": np.nan, "t_stat": np.nan,
                     "p_value": np.nan, "conf_low": np.nan, "conf_high": np.nan,
                     "significance_comment": comment})
    df = pd.DataFrame(rows)
    _, _, signs_ok = check_expected_signs(df, expected_signs)
    df["expected_sign"] = [expected_sign_for(v, expected_signs) for v in df["variable"]]
    df["sign_ok"] = signs_ok
    return df


def _backtest_regression(prepared_df, config, spec, meta, terms, tr, fit_predictor):
    test_size = config.get("test_size", 0) or 0
    n = len(prepared_df)
    if test_size <= 0 or n <= test_size + 5:
        return np.nan, np.nan, np.nan
    try:
        train_df = prepared_df.iloc[:-test_size].copy()
        train_df.attrs["meta"] = meta
        pred_train, _ = fit_predictor(train_df)
        n_hist = len(train_df)
        tb, tm = meta["target_base"], meta["target_modeled"]
        mod_init = prepared_df[tm].values.astype(float).copy(); mod_init[n_hist:] = np.nan
        lvl_init = prepared_df[tb].values.astype(float).copy(); lvl_init[n_hist:] = np.nan
        exog_series = {t["tcol"]: prepared_df[t["tcol"]].values.astype(float)
                       for t in terms if not t["is_y"]}
        meta_bt = {"terms": terms, "target_transform": tr, "predict_scalar": pred_train}
        _, lvl = recursive_linreg_forecast(meta_bt, exog_series, mod_init, lvl_init, n_hist)
        actual = prepared_df[tb].values[n_hist:]
        return _rmse(actual, lvl[n_hist:]), _mae(actual, lvl[n_hist:]), _mape(actual, lvl[n_hist:])
    except Exception:
        return np.nan, np.nan, np.nan


def _fit_regression(prepared_df, config, spec, model_type, mode):
    """OLS-ailesi (mode='ols') ve sklearn-ailesi (mode='sklearn:*') ortak fit'i."""
    meta = prepared_df.attrs["meta"]
    tr, tb = meta["target_transform"], meta["target_base"]
    exp_signs = config.get("expected_signs", {})

    bm = build_regression_matrix(prepared_df, spec, config)
    y_full, X_full, terms, add_c = bm["y"], bm["X"], bm["terms"], bm["add_constant"]
    y, X = align_xy(y_full, X_full)
    if X.shape[1] == 0 and not add_c:
        raise ValueError("Model icin regresor bulunamadi (exog/lags bos).")
    feat_cols = list(X.columns)

    def fit_predictor(df):
        bmi = build_regression_matrix(df, spec, config)
        yy, XX = align_xy(bmi["y"], bmi["X"])
        if feat_cols:
            XX = XX.reindex(columns=feat_cols)
        if mode == "ols":
            Xc = sm.add_constant(XX, has_constant="add") if add_c else XX
            r = sm.OLS(yy, Xc).fit(**_cov_kwargs(config))
            return make_linreg_predictor(r.params.to_dict(), add_c), r
        else:
            pipe = _cv_fit_sklearn(mode, XX.values, yy.values, config, spec)
            return make_sklearn_predictor(pipe, feat_cols), pipe

    predict_full, fit_full = fit_predictor(prepared_df)

    if mode == "ols":
        res = fit_full
        coef_df = coef_table_from_sm(res, exp_signs)
        diag = full_diagnostics(res, config)
        r2, adj = float(res.rsquared), float(res.rsquared_adj)
        aic, bic, llf, nobs = float(res.aic), float(res.bic), float(res.llf), int(res.nobs)
        fitted_mod = res.fittedvalues
        fit_obj = res
    else:
        pipe = fit_full
        Xc = sm.add_constant(X, has_constant="add")
        aux = sm.OLS(y, Xc).fit()
        diag = full_diagnostics(aux, config)
        coef_df = coef_table_sklearn(pipe, feat_cols, add_c, exp_signs)
        fitted_mod = pd.Series(pipe.predict(X.values), index=X.index)
        resid = y.values - fitted_mod.values
        rss = float(np.sum(resid ** 2))
        sst = float(np.sum((y.values - y.values.mean()) ** 2))
        r2 = 1 - rss / sst if sst > 0 else np.nan
        nobs = len(y)
        k = int(np.sum(np.abs(coef_df.loc[coef_df["variable"] != "const", "coef"]) > 1e-10)) + (1 if add_c else 0)
        aic, bic = _gaussian_ic(rss, nobs, k)
        adj = 1 - (1 - r2) * (nobs - 1) / (nobs - k - 1) if (nobs - k - 1) > 0 else np.nan
        llf = np.nan
        fit_obj = pipe

    vif = compute_vif(X)
    max_vif = float(vif["VIF"].max()) if len(vif) > 0 else np.nan
    tr3 = _level_train_metrics(fitted_mod, X.index, prepared_df, tr, tb)
    te3 = _backtest_regression(prepared_df, config, spec, meta, terms, tr, fit_predictor)

    exp_ok, _, _ = check_expected_signs(coef_df, exp_signs)
    sig_ratio = significant_coef_ratio(coef_df)
    insig = (1 - sig_ratio) if not np.isnan(sig_ratio) else np.nan
    clean, reasons = calculate_clean_score(diag, config, exp_ok, insig, te3[2], max_vif)

    metrics = _metrics_dict(r2, adj, aic, bic, llf, nobs, tr3, te3, max_vif, sig_ratio, exp_ok, clean)
    comments = build_comments(diag, vif, config, model_type,
                              extra={"clean_reasons": "; ".join(reasons) if reasons else "temiz"})
    meta_lr = {"kind": "linreg", "predict_scalar": predict_full, "terms": terms,
               "target_transform": tr, "add_constant": add_c, "feat_cols": feat_cols}
    return _make_result(model_type, model_type, fit_obj, coef_df, diag, vif, metrics, comments, meta_lr)


def fit_linreg_family(model_name, prepared_df, config, spec):
    return _fit_regression(prepared_df, config, spec, model_name, "ols")

### 5.5 ARDL (otomatik lag seçimi)

`max_lag_y` / `max_lag_x` sınırları içinde AIC/BIC ile en iyi lag kombinasyonu
aranır; seçilen yapı ADL biçiminde kurulur (böylece tanı testleri ve forecast
altyapısı aynen kullanılır). statsmodels `ARDL` mevcutsa çekirdek doğrulama
için kullanılabilir.

In [ ]:
def fit_ardl(prepared_df, config, spec):
    exog = spec.get("exog", [])
    maxp, maxq = int(spec.get("max_lag_y", 4)), int(spec.get("max_lag_x", 4))
    crit = spec.get("selection_criterion", "bic")
    best = None
    for p in range(0, maxp + 1):
        for q in range(0, maxq + 1):
            trial = {"exog": exog, "lags_y": list(range(1, p + 1)),
                     "lags_x": {b: list(range(0, q + 1)) for b in exog}, "add_constant": True}
            try:
                bm = build_regression_matrix(prepared_df, trial, config)
                yy, XX = align_xy(bm["y"], bm["X"])
                if len(yy) < XX.shape[1] + 3:
                    continue
                r = sm.OLS(yy, sm.add_constant(XX, has_constant="add")).fit()
                ic = float(r.bic) if crit == "bic" else float(r.aic)
                if best is None or ic < best[0]:
                    best = (ic, p, q, trial)
            except Exception:
                continue
    if best is None:
        raise ValueError("ARDL icin uygun lag kombinasyonu bulunamadi (ornek cok kucuk olabilir).")
    _, p, q, trial = best
    res = _fit_regression(prepared_df, config, trial, "ARDL", "ols")
    res["comments"]["ardl"] = f"ARDL lag secimi ({crit.upper()}): p(y)={p}, q(x)={q}."
    res["metrics"]["ardl_p"], res["metrics"]["ardl_q"] = p, q
    return res

### 5.6 ECM (Engle-Granger, iki aşamalı)

1) Uzun dönem: `y = c + b·x + u` (seviyelerde).
2) Kısa dönem: `Δy = c + λ·u_{t-1} + Δy lagleri + Δx lagleri + e`.
`λ` (hata düzeltme terimi) negatif ve anlamlı ise mekanizma çalışıyor demektir.

In [ ]:
def _ecm_estimate(df, spec, config):
    meta = df.attrs["meta"]
    tb = meta["target_base"]
    level_vars = spec.get("level_vars", [])
    diff_vars = spec.get("diff_vars", [])

    lr_X = pd.DataFrame({v: df[v] for v in level_vars}, index=df.index)
    yy, XX = align_xy(df[tb], lr_X)
    lr_res = sm.OLS(yy, sm.add_constant(XX, has_constant="add")).fit()
    u = df[tb] - lr_res.predict(sm.add_constant(lr_X, has_constant="add"))

    dy = df[tb].diff()
    sr = pd.DataFrame(index=df.index)
    sr["ECT_L1"] = u.shift(1)
    for L in spec.get("lags_y", [1]):
        sr[f"L{L}_d_{tb}"] = dy.shift(L)
    dx_terms = []
    for base in diff_vars:
        dxs = df[base].diff()
        for L in spec.get("lags_x", {}).get(base, [0]):
            nm = f"d_{base}" if L == 0 else f"L{L}_d_{base}"
            sr[nm] = dxs.shift(L)
            dx_terms.append({"base": base, "lag": L, "name": nm})
    dyy, srX = align_xy(dy, sr)
    sr_res = sm.OLS(dyy, sm.add_constant(srX, has_constant="add")).fit(**_cov_kwargs(config))

    meta_ecm = {"kind": "ecm", "lr_params": lr_res.params.to_dict(), "level_vars": level_vars,
                "sr_params": sr_res.params.to_dict(), "lags_y": spec.get("lags_y", [1]),
                "dx_terms": dx_terms, "target_base": tb, "add_constant": True}
    return lr_res, sr_res, srX, dyy, u, meta_ecm


def ecm_forecast(meta, x_level, y_level_init, n_hist):
    lrp, sp = meta["lr_params"], meta["sr_params"]
    lv, lags_y, dxt, tb = meta["level_vars"], meta["lags_y"], meta["dx_terms"], meta["target_base"]
    y = np.array(y_level_init, float)
    N = len(y)

    def u_at(i):
        val = lrp.get("const", 0.0)
        for v in lv:
            val += lrp.get(v, 0.0) * x_level[v][i]
        return y[i] - val

    for i in range(n_hist, N):
        if np.isnan(y[i - 1]):
            y[i] = np.nan; continue
        pred = sp.get("const", 0.0) + sp.get("ECT_L1", 0.0) * u_at(i - 1)
        for L in lags_y:
            j = i - L
            dyv = 0.0 if j - 1 < 0 else (y[j] - y[j - 1])
            pred += sp.get(f"L{L}_d_{tb}", 0.0) * dyv
        for t in dxt:
            j = i - t["lag"]
            dxv = 0.0 if j - 1 < 0 else (x_level[t["base"]][j] - x_level[t["base"]][j - 1])
            pred += sp.get(t["name"], 0.0) * dxv
        y[i] = y[i - 1] + pred
    return y


def fit_ecm(prepared_df, config, spec):
    meta = prepared_df.attrs["meta"]
    tb = meta["target_base"]
    exp_signs = config.get("expected_signs", {})
    lr_res, sr_res, srX, dyy, u, meta_ecm = _ecm_estimate(prepared_df, spec, config)

    # Engle-Granger: uzun dönem artığına ADF
    try:
        adf = adfuller(u.dropna(), autolag="AIC")
        eg_p = float(adf[1])
    except Exception:
        eg_p = np.nan

    coef_sr = coef_table_from_sm(sr_res, exp_signs)
    coef_lr = coef_table_from_sm(lr_res, exp_signs)
    coef_lr["variable"] = ["LR_" + v for v in coef_lr["variable"]]
    coef_all = pd.concat([coef_sr, coef_lr], ignore_index=True)

    ect = sr_res.params.get("ECT_L1", np.nan)
    ect_p = sr_res.pvalues.get("ECT_L1", np.nan)
    if not np.isnan(ect) and ect < 0 and not np.isnan(ect_p) and ect_p < 0.05:
        ect_comment = f"Hata duzeltme terimi lambda={ect:.3f} negatif ve anlamli (p={ect_p:.3f}): ECM mekanizmasi calisiyor."
    elif not np.isnan(ect) and ect < 0:
        ect_comment = f"Hata duzeltme terimi lambda={ect:.3f} negatif fakat zayif/anlamsiz (p={ect_p:.3f})."
    else:
        ect_comment = "Hata duzeltme terimi negatif degil: ECM mekanizmasi teorik olarak zayif veya hatali olabilir."
    eg_comment = ("Engle-Granger: uzun donem iliskisi esbutunlesik (durgun artik)."
                  if (not np.isnan(eg_p) and eg_p < 0.05)
                  else "Engle-Granger: esbutunlesme kaniti zayif (artik durgun olmayabilir).")

    diag = full_diagnostics(sr_res, config)
    vif = compute_vif(srX)
    max_vif = float(vif["VIF"].max()) if len(vif) > 0 else np.nan

    # Metrikler (Δy -> seviye)
    fitted_dy = sr_res.fittedvalues
    idx = fitted_dy.index
    prev = prepared_df[tb].shift(1).reindex(idx).values
    actual_lvl = prepared_df[tb].reindex(idx).values
    fit_lvl = prev + fitted_dy.values
    tr3 = (_rmse(actual_lvl, fit_lvl), _mae(actual_lvl, fit_lvl), _mape(actual_lvl, fit_lvl))

    # Backtest
    te3 = (np.nan, np.nan, np.nan)
    test_size = config.get("test_size", 0) or 0
    if test_size > 0 and len(prepared_df) > test_size + 5:
        try:
            train_df = prepared_df.iloc[:-test_size].copy(); train_df.attrs["meta"] = meta
            _, _, _, _, _, meta_bt = _ecm_estimate(train_df, spec, config)
            n_hist = len(train_df)
            x_level = {v: prepared_df[v].values.astype(float) for v in meta_ecm["level_vars"]}
            y_init = prepared_df[tb].values.astype(float).copy(); y_init[n_hist:] = np.nan
            y_init[n_hist - 1] = prepared_df[tb].values[n_hist - 1]
            yhat = ecm_forecast(meta_bt, x_level, y_init, n_hist)
            actual = prepared_df[tb].values[n_hist:]
            te3 = (_rmse(actual, yhat[n_hist:]), _mae(actual, yhat[n_hist:]), _mape(actual, yhat[n_hist:]))
        except Exception:
            pass

    r2, adj = float(sr_res.rsquared), float(sr_res.rsquared_adj)
    aic, bic, llf, nobs = float(sr_res.aic), float(sr_res.bic), float(sr_res.llf), int(sr_res.nobs)
    exp_ok, _, _ = check_expected_signs(coef_all, exp_signs)
    sig_ratio = significant_coef_ratio(coef_sr)
    insig = (1 - sig_ratio) if not np.isnan(sig_ratio) else np.nan
    clean, reasons = calculate_clean_score(diag, config, exp_ok, insig, te3[2], max_vif)

    metrics = _metrics_dict(r2, adj, aic, bic, llf, nobs, tr3, te3, max_vif, sig_ratio, exp_ok, clean)
    metrics["eg_pvalue"] = eg_p
    metrics["ect_coef"] = float(ect) if not np.isnan(ect) else np.nan
    metrics["ect_pvalue"] = float(ect_p) if not np.isnan(ect_p) else np.nan
    comments = build_comments(diag, vif, config, "ECM",
                              extra={"ect": ect_comment, "cointegration": eg_comment,
                                     "clean_reasons": "; ".join(reasons) if reasons else "temiz"})
    return _make_result("ECM", "ECM", sr_res, coef_all, diag, vif, metrics, comments, meta_ecm)

### 5.7 VAR / 5.8 VECM / 5.9 SARIMAX / Ridge-Lasso-ElasticNet

- **VAR**: lag seçim tablosu, stabilite kontrolü, opsiyonel Granger nedensellik.
- **VECM**: Johansen eşbütünleşme testi, alpha (uyarlama) ve beta (uzun dönem)
  katsayıları.
- **SARIMAX**: `order`, `seasonal_order`, `exog` CONFIG'ten.
- **Ridge/Lasso/ElasticNet**: StandardScaler + TimeSeriesSplit CV ile alpha
  seçimi; Lasso'da sıfıra düşen katsayılar ayrıca raporlanır.

Not: VAR/VECM forecast'ları içsel (senaryodan bağımsız) üretilir; diğer
değişkenler sistem tarafından tahmin edilir.

In [ ]:
def reconstruct_levels(modeled, last_level, tr):
    out = np.empty(len(modeled), float)
    prev = last_level
    for k, m in enumerate(modeled):
        out[k] = float(invert_transform_step(prev, m, tr))
        prev = out[k]
    return out


# ---------------- SARIMAX ----------------
def fit_sarimax(prepared_df, config, spec):
    if not HAVE.get("sarimax"):
        raise RuntimeError("statsmodels SARIMAX bileseni yok.")
    meta = prepared_df.attrs["meta"]
    tr, tb, tm = meta["target_transform"], meta["target_base"], meta["target_modeled"]
    exog_bases = spec.get("exog", [])
    exog_cols = [meta["base_to_modeled"].get(b, b) for b in exog_bases]
    endog = prepared_df[tm]
    if exog_cols:
        data = pd.concat([endog.rename("__y__"), prepared_df[exog_cols]], axis=1).dropna()
        endog_a, exog_a = data["__y__"], data.drop(columns="__y__")
    else:
        endog_a, exog_a = endog.dropna(), None
    order = tuple(spec.get("order", [1, 0, 1]))
    sorder = tuple(spec.get("seasonal_order", [0, 0, 0, 0]))
    res = SARIMAX(endog_a, exog=exog_a, order=order, seasonal_order=sorder,
                  enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)

    coef_df = coef_table_from_sm(res, config.get("expected_signs", {}))
    diag = residual_only_diagnostics(res.resid)

    idx = endog_a.index
    prev = prepared_df[tb].shift(1).reindex(idx).values
    fit_lvl = invert_transform_array(prev, res.fittedvalues.reindex(idx).values, tr)
    actual_lvl = prepared_df[tb].reindex(idx).values
    tr3 = (_rmse(actual_lvl, fit_lvl), _mae(actual_lvl, fit_lvl), _mape(actual_lvl, fit_lvl))

    # R2 seviye uzayında, durum-uzayı ısınma (burn-in) gözlemleri atlanarak
    burn = max(3, int(order[0]) + (int(sorder[0]) * int(sorder[3]) if len(sorder) > 3 else 0))
    a_r, f_r = actual_lvl[burn:], fit_lvl[burn:]
    sst = float(np.nansum((a_r - np.nanmean(a_r)) ** 2))
    r2 = 1 - float(np.nansum((a_r - f_r) ** 2)) / sst if sst > 0 else np.nan
    aic, bic, llf, nobs = float(res.aic), float(res.bic), float(res.llf), int(res.nobs)

    te3 = (np.nan, np.nan, np.nan)
    test_size = config.get("test_size", 0) or 0
    if test_size > 0 and len(endog_a) > test_size + 5:
        try:
            tr_endog = endog_a.iloc[:-test_size]
            tr_exog = exog_a.iloc[:-test_size] if exog_a is not None else None
            te_exog = exog_a.iloc[-test_size:] if exog_a is not None else None
            rt = SARIMAX(tr_endog, exog=tr_exog, order=order, seasonal_order=sorder,
                         enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
            fc_mod = np.asarray(rt.get_forecast(steps=test_size, exog=te_exog).predicted_mean)
            last_level = prepared_df[tb].reindex(endog_a.index).values[-test_size - 1]
            fc_lvl = reconstruct_levels(fc_mod, last_level, tr)
            actual = prepared_df[tb].reindex(endog_a.index).values[-test_size:]
            te3 = (_rmse(actual, fc_lvl), _mae(actual, fc_lvl), _mape(actual, fc_lvl))
        except Exception:
            pass

    exp_ok, _, _ = check_expected_signs(coef_df, config.get("expected_signs", {}))
    sig_ratio = significant_coef_ratio(coef_df)
    insig = (1 - sig_ratio) if not np.isnan(sig_ratio) else np.nan
    clean, reasons = calculate_clean_score(diag, config, exp_ok, insig, te3[2], np.nan)
    metrics = _metrics_dict(r2, np.nan, aic, bic, llf, nobs, tr3, te3, np.nan, sig_ratio, exp_ok, clean)
    comments = interpret_diagnostics(diag)
    comments["model"] = f"SARIMAX order={order}, seasonal={sorder}, exog={exog_bases}."
    comments["clean_reasons"] = "; ".join(reasons) if reasons else "temiz"
    meta_sx = {"kind": "sarimax", "res": res, "exog_cols": exog_cols, "target_transform": tr,
               "target_base": tb, "order": order, "seasonal_order": sorder}
    return _make_result("SARIMAX", "SARIMAX", res, coef_df, diag, pd.DataFrame(), metrics, comments, meta_sx)


# ---------------- VAR ----------------
def var_coef_table(res, target, config):
    exp = config.get("expected_signs", {})
    params, ses = res.params[target], res.stderr[target]
    tv, pv = res.tvalues[target], res.pvalues[target]
    rows = []
    for name in params.index:
        co, se, p = float(params[name]), float(ses[name]), float(pv[name])
        rows.append({"variable": str(name), "coef": co, "std_error": se, "t_stat": float(tv[name]),
                     "p_value": p, "conf_low": co - 1.96 * se, "conf_high": co + 1.96 * se,
                     "significance_comment": interpret_pvalue(p)})
    df = pd.DataFrame(rows)
    _, _, signs_ok = check_expected_signs(df, exp)
    df["expected_sign"] = [expected_sign_for(v, exp) for v in df["variable"]]
    df["sign_ok"] = signs_ok
    return df


def fit_var(prepared_df, config, spec):
    if not HAVE.get("var"):
        raise RuntimeError("statsmodels VAR bileseni yok.")
    meta = prepared_df.attrs["meta"]
    vars_ = spec.get("vars", [])
    if len(vars_) < 2:
        raise ValueError("VAR icin en az 2 degisken gerekir (CONFIG['model_specs']['VAR']['vars']).")
    endog = prepared_df[vars_].dropna()
    target = meta["target_base"] if meta["target_base"] in vars_ else vars_[0]
    tpos = vars_.index(target)
    model = VAR(endog)
    mm = max(1, min(int(spec.get("maxlags", 6)), len(endog) // 3))

    sel_rows = []
    for L in range(1, mm + 1):
        try:
            rL = model.fit(L)
            sel_rows.append({"lag": L, "AIC": float(rL.aic), "BIC": float(rL.bic),
                             "HQIC": float(rL.hqic), "FPE": float(rL.fpe)})
        except Exception:
            pass
    sel_df = pd.DataFrame(sel_rows)
    ic = spec.get("ic", "bic").upper()
    if len(sel_df) > 0 and ic in sel_df.columns:
        best_lag = int(sel_df.loc[sel_df[ic].idxmin(), "lag"])
    else:
        best_lag = 1
    res = model.fit(best_lag)
    k = res.k_ar

    coef_df = var_coef_table(res, target, config)
    diag = residual_only_diagnostics(np.asarray(res.resid[target]))

    fitted = res.fittedvalues[target]
    idx = fitted.index
    actual = endog[target].reindex(idx)
    tr3 = (_rmse(actual, fitted), _mae(actual, fitted), _mape(actual, fitted))
    sst = float(np.sum((actual - actual.mean()) ** 2))
    r2 = 1 - float(np.sum((actual - fitted) ** 2)) / sst if sst > 0 else np.nan
    # AIC/BIC: hedef denklem artiklarindan, regresyon modelleriyle ayni olcekte
    resid_t = (actual - fitted).dropna().values
    n_t, k_t = len(resid_t), int(res.params[target].shape[0])
    aic, bic = _gaussian_ic(float(np.sum(resid_t ** 2)), n_t, k_t)
    llf, nobs = float(res.llf), int(res.nobs)

    te3 = (np.nan, np.nan, np.nan)
    test_size = config.get("test_size", 0) or 0
    if test_size > 0 and len(endog) > test_size + k + 2:
        try:
            train = endog.iloc[:-test_size]
            rt = VAR(train).fit(k)
            fc = rt.forecast(train.values[-rt.k_ar:], steps=test_size)
            pred = fc[:, tpos]
            actual_te = endog[target].values[-test_size:]
            te3 = (_rmse(actual_te, pred), _mae(actual_te, pred), _mape(actual_te, pred))
        except Exception:
            pass

    stable = None
    try:
        stable = bool(res.is_stable())
    except Exception:
        pass
    granger = {}
    if spec.get("granger", False):
        for other in vars_:
            if other == target:
                continue
            try:
                gc = res.test_causality(target, [other], kind="f")
                granger[other] = float(gc.pvalue)
            except Exception:
                granger[other] = np.nan

    exp_ok, _, _ = check_expected_signs(coef_df, config.get("expected_signs", {}))
    sig_ratio = significant_coef_ratio(coef_df)
    insig = (1 - sig_ratio) if not np.isnan(sig_ratio) else np.nan
    clean, reasons = calculate_clean_score(diag, config, exp_ok, insig, te3[2], np.nan)
    metrics = _metrics_dict(r2, np.nan, aic, bic, llf, nobs, tr3, te3, np.nan, sig_ratio, exp_ok, clean)
    metrics["var_lag"] = k
    metrics["stable"] = stable

    comments = interpret_diagnostics(diag)
    comments["model"] = f"VAR({k}) degiskenler={vars_}; hedef={target}."
    comments["stability"] = ("Sistem stabil (birim kok yok)." if stable else
                             ("Sistem stabil degil (dikkat)." if stable is not None else "Stabilite hesaplanamadi."))
    if granger:
        gtxt = ", ".join([f"{o}: p={p:.3f}" for o, p in granger.items()])
        comments["granger"] = f"Granger nedensellik ({target} uzerine) -> {gtxt}"
    comments["forecast_note"] = "VAR forecast'lari icseldir; forecast senaryolari uygulanmaz."
    comments["clean_reasons"] = "; ".join(reasons) if reasons else "temiz"

    meta_var = {"kind": "var", "res": res, "vars": vars_, "target_pos": tpos,
                "k_ar": k, "target_base": target}
    result = _make_result("VAR", "VAR", res, coef_df, diag, pd.DataFrame(), metrics, comments, meta_var)
    result["_meta"]["select_order"] = sel_df
    return result


# ---------------- VECM ----------------
def vecm_coef_table(res, vars_, target, config):
    exp = config.get("expected_signs", {})
    tpos = vars_.index(target)
    rows = []
    alpha = np.atleast_2d(res.alpha)
    try:
        alpha_p = np.atleast_2d(res.pvalues_alpha)
    except Exception:
        alpha_p = None
    rank = alpha.shape[1]
    for r in range(rank):
        co = float(alpha[tpos, r])
        p = float(alpha_p[tpos, r]) if alpha_p is not None else np.nan
        rows.append({"variable": f"alpha_ec{r + 1}", "coef": co, "std_error": np.nan, "t_stat": np.nan,
                     "p_value": p, "conf_low": np.nan, "conf_high": np.nan,
                     "significance_comment": interpret_pvalue(p)})
    beta = np.atleast_2d(res.beta)
    for r in range(beta.shape[1]):
        for vi, v in enumerate(vars_):
            if vi >= beta.shape[0]:
                break
            rows.append({"variable": f"beta_ec{r + 1}_{v}", "coef": float(beta[vi, r]),
                         "std_error": np.nan, "t_stat": np.nan, "p_value": np.nan,
                         "conf_low": np.nan, "conf_high": np.nan,
                         "significance_comment": "esbutunlesme (uzun donem) katsayisi"})
    df = pd.DataFrame(rows)
    df["expected_sign"] = [expected_sign_for(v, exp) for v in df["variable"]]
    df["sign_ok"] = [None] * len(df)
    return df


def fit_vecm(prepared_df, config, spec):
    if not HAVE.get("var"):
        raise RuntimeError("statsmodels VECM bileseni yok.")
    meta = prepared_df.attrs["meta"]
    vars_ = spec.get("vars", [])
    if len(vars_) < 2:
        raise ValueError("VECM icin en az 2 degisken gerekir.")
    endog = prepared_df[vars_].dropna()
    target = meta["target_base"] if meta["target_base"] in vars_ else vars_[0]
    tpos = vars_.index(target)
    k_ar_diff = int(spec.get("k_ar_diff", 1))
    rank = int(spec.get("coint_rank", 1))
    det = spec.get("deterministic", "ci")

    johansen_txt = "Johansen testi hesaplanamadi."
    try:
        joh = coint_johansen(endog.values, 0, k_ar_diff)
        n_ce = int(np.sum(joh.lr1 > joh.cvt[:, 1]))  # %5 kritik değere göre
        johansen_txt = (f"Johansen (trace, %5): tahmini esbutunlesme sayisi = {n_ce}. "
                        f"CONFIG coint_rank={rank}.")
    except Exception:
        pass

    res = VECM(endog, k_ar_diff=k_ar_diff, coint_rank=rank, deterministic=det).fit()
    coef_df = vecm_coef_table(res, vars_, target, config)

    resid = np.asarray(res.resid)
    diag = residual_only_diagnostics(resid[:, tpos])
    n_res = resid.shape[0]
    actual_tail = endog[target].values[-n_res:]
    fitted_tail = actual_tail - resid[:, tpos]
    tr3 = (_rmse(actual_tail, fitted_tail), _mae(actual_tail, fitted_tail), _mape(actual_tail, fitted_tail))
    sst = float(np.sum((actual_tail - actual_tail.mean()) ** 2))
    r2 = 1 - float(np.sum(resid[:, tpos] ** 2)) / sst if sst > 0 else np.nan
    llf = float(getattr(res, "llf", np.nan))
    # AIC/BIC: hedef denklem artiklarindan, diger modellerle ayni olcekte
    k_t = len(vars_) * k_ar_diff + rank + 1
    aic_v, bic_v = _gaussian_ic(float(np.sum(resid[:, tpos] ** 2)), n_res, k_t)

    te3 = (np.nan, np.nan, np.nan)
    test_size = config.get("test_size", 0) or 0
    if test_size > 0 and len(endog) > test_size + k_ar_diff + 3:
        try:
            train = endog.iloc[:-test_size]
            rt = VECM(train, k_ar_diff=k_ar_diff, coint_rank=rank, deterministic=det).fit()
            fc = rt.predict(steps=test_size)
            pred = np.asarray(fc)[:, tpos]
            actual_te = endog[target].values[-test_size:]
            te3 = (_rmse(actual_te, pred), _mae(actual_te, pred), _mape(actual_te, pred))
        except Exception:
            pass

    alpha_t = float(np.atleast_2d(res.alpha)[tpos, 0])
    if alpha_t < 0:
        alpha_comment = f"Hedef denklemin alpha (uyarlama) katsayisi {alpha_t:.3f} < 0: uzun donem dengeye geri donus var."
    else:
        alpha_comment = f"Hedef denklemin alpha katsayisi {alpha_t:.3f} >= 0: uyarlama mekanizmasi zayif/ters olabilir."

    exp_ok, _, _ = check_expected_signs(coef_df, config.get("expected_signs", {}))
    clean, reasons = calculate_clean_score(diag, config, exp_ok, np.nan, te3[2], np.nan)
    metrics = _metrics_dict(r2, np.nan, aic_v, bic_v, llf, n_res, tr3, te3, np.nan, np.nan, exp_ok, clean)
    metrics["coint_rank"] = rank
    metrics["alpha_target"] = alpha_t

    comments = interpret_diagnostics(diag)
    comments["model"] = f"VECM(k_ar_diff={k_ar_diff}, rank={rank}, det='{det}') degiskenler={vars_}."
    comments["johansen"] = johansen_txt
    comments["alpha"] = alpha_comment
    comments["forecast_note"] = "VECM forecast'lari icseldir; forecast senaryolari uygulanmaz."
    comments["clean_reasons"] = "; ".join(reasons) if reasons else "temiz"

    meta_vecm = {"kind": "vecm", "res": res, "vars": vars_, "target_pos": tpos, "target_base": target}
    return _make_result("VECM", "VECM", res, coef_df, diag, pd.DataFrame(), metrics, comments, meta_vecm)


# ---------------- Ridge / Lasso / ElasticNet ----------------
def fit_sklearn_family(model_name, prepared_df, config, spec):
    mode = {"RIDGE": "sklearn:ridge", "LASSO": "sklearn:lasso", "ELASTICNET": "sklearn:en"}[model_name]
    res = _fit_regression(prepared_df, config, spec, model_name, mode)
    pipe = res["fit_object"]
    a = getattr(pipe, "_best_alpha", None)
    l1 = getattr(pipe, "_best_l1", None)
    res["comments"]["cv"] = f"CV ile secilen alpha={a}" + (f", l1_ratio={l1}" if l1 else "")
    res["metrics"]["best_alpha"] = a
    if l1:
        res["metrics"]["best_l1"] = l1
    if model_name == "LASSO":
        cf = res["coefficients"]
        zeros = cf[(cf["variable"] != "const") & (cf["coef"].abs() < 1e-10)]["variable"].tolist()
        res["comments"]["lasso_zero"] = ("Sifira dusen katsayilar: " + ", ".join(zeros)) if zeros else "Sifira dusen katsayi yok."
    return res


# ---------------- Dispatcher ----------------
def run_model(model_name, prepared_df, config):
    spec = config["model_specs"].get(model_name, {})
    mt = model_name
    if model_name in ("OLS", "ADL", "DISTRIBUTED_LAG"):
        return safe_fit_model(lambda: fit_linreg_family(model_name, prepared_df, config, spec), model_name, mt)
    if model_name == "ARDL":
        return safe_fit_model(lambda: fit_ardl(prepared_df, config, spec), model_name, mt)
    if model_name == "ECM":
        return safe_fit_model(lambda: fit_ecm(prepared_df, config, spec), model_name, mt)
    if model_name == "VAR":
        return safe_fit_model(lambda: fit_var(prepared_df, config, spec), model_name, mt)
    if model_name == "VECM":
        return safe_fit_model(lambda: fit_vecm(prepared_df, config, spec), model_name, mt)
    if model_name == "SARIMAX":
        return safe_fit_model(lambda: fit_sarimax(prepared_df, config, spec), model_name, mt)
    if model_name in ("RIDGE", "LASSO", "ELASTICNET"):
        return safe_fit_model(lambda: fit_sklearn_family(model_name, prepared_df, config, spec), model_name, mt)
    return make_failed_result(model_name, mt, "Bilinmeyen model turu.")

## 6. Forecast modülü

Forecast, `CONFIG["forecast_scenarios"]` içindeki aktif senaryoya göre çalışır.
Desteklenen senaryo tipleri: `constant`, `growth`, `path`, `linear`, `shock`.
Bağımsız değişken için senaryo tanımlanmamışsa son gözlem sabit tutulur (uyarı
verilir). Lagli modeller özyinelemeli (recursive) tahmin eder; log/diff/logdiff
modellerinde sonuç otomatik olarak seviyeye geri çevrilir.

In [ ]:
def get_active_scenario(config):
    return config["forecast_scenarios"].get(config.get("active_scenario", "baseline"), {})


def build_future_index(prepared_df, config):
    freq = prepared_df.attrs["meta"]["freq"]
    last = prepared_df.index[-1]
    fs, fe = config.get("forecast_start"), config.get("forecast_end")
    start = pd.date_range(last, periods=2, freq=freq)[1] if fs is None else pd.Timestamp(fs)
    if fe is None:
        h = int(config.get("forecast_horizon", 12))
        return pd.date_range(start, periods=h, freq=freq)
    return pd.date_range(start, pd.Timestamp(fe), freq=freq)


def scenario_level_series(base, scen, future_index, last_level, warns):
    spec = scen.get(base)
    n = len(future_index)
    if spec is None:
        warns.append(f"'{base}' icin senaryo tanimlanmadi; son gozlem ({last_level:.4g}) sabit tutuldu.")
        return pd.Series([last_level] * n, index=future_index)
    typ = spec.get("type")
    if typ == "constant":
        return pd.Series([spec["value"]] * n, index=future_index)
    if typ == "growth":
        r = spec.get("monthly_rate", spec.get("rate", 0.0))
        return pd.Series([last_level * ((1 + r) ** (k + 1)) for k in range(n)], index=future_index)
    if typ == "linear":
        return pd.Series(np.linspace(spec["start"], spec["end"], n), index=future_index)
    if typ == "path":
        s = pd.Series({pd.Timestamp(k): float(v) for k, v in spec.get("values", {}).items()})
        s = s.reindex(future_index.union(s.index)).interpolate().reindex(future_index).ffill().bfill()
        if s.isna().all():
            s = pd.Series([last_level] * n, index=future_index)
        return s
    if typ == "shock":
        sd = pd.Timestamp(spec.get("shock_date"))
        g, sh = spec.get("after_shock_growth", 0.0), spec.get("shock_size", 0.0)
        vals, lvl, done = [], last_level, False
        for d in future_index:
            lvl = lvl * (1 + g)
            if (not done) and d >= sd:
                lvl = lvl * (1 + sh); done = True
            vals.append(lvl)
        return pd.Series(vals, index=future_index)
    warns.append(f"'{base}' icin bilinmeyen senaryo tipi '{typ}'; son gozlem sabit tutuldu.")
    return pd.Series([last_level] * n, index=future_index)


def build_forecast_inputs(prepared_df, config, future_index):
    """Gelecek dönem için seviye ve transform edilmiş bağımsız değişken serileri."""
    meta = prepared_df.attrs["meta"]
    scen = get_active_scenario(config)
    warns = []
    combined = prepared_df.index.append(future_index)

    bases = set(config.get("exog_transforms", {}).keys()) | set(scen.keys())
    for _n, sp in config["model_specs"].items():
        bases |= set(sp.get("exog", []))
        bases |= set(sp.get("level_vars", []))
        bases |= set(sp.get("diff_vars", []))

    level_full, trans_full = {}, {}
    for b in bases:
        if b not in prepared_df.columns:
            continue
        last_level = float(prepared_df[b].dropna().iloc[-1])
        fut = scenario_level_series(b, scen, future_index, last_level, warns)
        full_level = pd.concat([prepared_df[b], fut])
        full_level = full_level[~full_level.index.duplicated(keep="last")].reindex(combined)
        level_full[b] = full_level.values.astype(float)
        tcol = meta["base_to_modeled"].get(b, b)
        reg_tr = meta["registry"].get(tcol, {}).get("transform", "level")
        try:
            trans_full[tcol] = apply_transform(full_level, reg_tr).values.astype(float)
        except Exception:
            trans_full[tcol] = full_level.values.astype(float)
    return combined, level_full, trans_full, warns


def run_forecasts(results, prepared_df, config):
    meta = prepared_df.attrs["meta"]
    tr, tb, tm = meta["target_transform"], meta["target_base"], meta["target_modeled"]
    future_index = build_future_index(prepared_df, config)
    combined, level_full, trans_full, warns = build_forecast_inputs(prepared_df, config, future_index)
    n_hist, N = len(prepared_df), len(combined)
    out = {"future_index": future_index, "scenario_warnings": warns, "forecasts": {}, "errors": {}}

    for res in results:
        if res["status"] != "success":
            continue
        m = res.get("_meta", {})
        kind = m.get("kind")
        name = res["model_name"]
        try:
            if kind == "linreg":
                mod_init = np.full(N, np.nan); lvl_init = np.full(N, np.nan)
                mod_init[:n_hist] = prepared_df[tm].values
                lvl_init[:n_hist] = prepared_df[tb].values
                exog_series = {t["tcol"]: trans_full.get(t["tcol"]) for t in m["terms"] if not t["is_y"]}
                meta_lr = {"terms": m["terms"], "target_transform": tr, "predict_scalar": m["predict_scalar"]}
                _, lvl = recursive_linreg_forecast(meta_lr, exog_series, mod_init, lvl_init, n_hist)
                series = pd.Series(lvl[n_hist:], index=future_index)
            elif kind == "ecm":
                x_level = {v: level_full.get(v) for v in m["level_vars"]}
                y_init = np.full(N, np.nan); y_init[:n_hist] = prepared_df[tb].values
                yhat = ecm_forecast(m, x_level, y_init, n_hist)
                series = pd.Series(yhat[n_hist:], index=future_index)
            elif kind == "sarimax":
                res_sx = m["res"]; h = len(future_index)
                if m["exog_cols"]:
                    fx = np.column_stack([trans_full.get(c)[n_hist:] for c in m["exog_cols"]])
                    fc = res_sx.get_forecast(steps=h, exog=fx).predicted_mean
                else:
                    fc = res_sx.get_forecast(steps=h).predicted_mean
                last_level = float(prepared_df[tb].dropna().iloc[-1])
                series = pd.Series(reconstruct_levels(np.asarray(fc), last_level, tr), index=future_index)
            elif kind == "var":
                res_v, k, vars_, tpos = m["res"], m["k_ar"], m["vars"], m["target_pos"]
                endog = prepared_df[vars_].dropna()
                fc = res_v.forecast(endog.values[-k:], steps=len(future_index))
                series = pd.Series(fc[:, tpos], index=future_index)
            elif kind == "vecm":
                res_v, tpos = m["res"], m["target_pos"]
                fc = np.asarray(res_v.predict(steps=len(future_index)))
                series = pd.Series(fc[:, tpos], index=future_index)
            else:
                continue
            res["forecast"] = series.to_frame("forecast")
            out["forecasts"][name] = series
        except Exception as e:
            out["errors"][name] = f"{type(e).__name__}: {e}"
    return out


def attach_selected_forecast(forecast_results, best_model):
    """En iyi model için 'selected_model_forecast' serisini ekler."""
    if best_model and best_model in forecast_results["forecasts"]:
        forecast_results["selected_model"] = best_model
        forecast_results["selected_model_forecast"] = forecast_results["forecasts"][best_model]
    else:
        forecast_results["selected_model"] = best_model
        forecast_results["selected_model_forecast"] = None
    return forecast_results

## 7. Model karşılaştırma ve en iyi model seçimi

Tüm modeller tek bir karşılaştırma tablosunda toplanır. Seçim kriteri
`CONFIG["selection_metric"]` ile belirlenir (bic/aic → en düşük; rmse/mae/mape
→ en düşük test hatası; adjusted_r2/clean_score → en yüksek). Seçilen model tanı
testlerinde sorunluysa rapor bunu açıkça belirtir.

In [ ]:
_SELECTION_MAP = {"bic": "bic", "aic": "aic", "rmse": "rmse_test", "mae": "mae_test",
                  "mape": "mape_test", "adjusted_r2": "adj_r2", "clean_score": "clean_score"}
_HIGHER_BETTER = {"adjusted_r2", "clean_score", "r2"}


def _selection_value(metrics, metric):
    return metrics.get(_SELECTION_MAP.get(metric, "bic"), np.nan)


def compare_models(results, config):
    metric = config.get("selection_metric", "bic")
    dep = transformed_name(config["target_col"], config.get("target_transform", "level"))
    rows = []
    for res in results:
        mt = res.get("metrics", {})
        diag = res.get("diagnostics", {})
        spec = config["model_specs"].get(res["model_name"], {})
        exogv = spec.get("exog") or spec.get("vars") or spec.get("level_vars") or []
        rows.append({
            "model_name": res["model_name"], "model_type": res["model_type"], "status": res["status"],
            "nobs": mt.get("nobs", np.nan), "dependent_variable": dep,
            "exog_variables": ", ".join(exogv) if exogv else "",
            "r2": mt.get("r2", np.nan), "adj_r2": mt.get("adj_r2", np.nan),
            "aic": mt.get("aic", np.nan), "bic": mt.get("bic", np.nan),
            "rmse_train": mt.get("rmse_train", np.nan), "mae_train": mt.get("mae_train", np.nan),
            "mape_train": mt.get("mape_train", np.nan), "rmse_test": mt.get("rmse_test", np.nan),
            "mae_test": mt.get("mae_test", np.nan), "mape_test": mt.get("mape_test", np.nan),
            "bg_pvalue": diag.get("bg_pvalue", np.nan), "ljungbox_pvalue": diag.get("ljungbox_pvalue", np.nan),
            "bp_pvalue": diag.get("bp_pvalue", np.nan), "white_pvalue": diag.get("white_pvalue", np.nan),
            "jb_pvalue": diag.get("jb_pvalue", np.nan), "reset_pvalue": diag.get("reset_pvalue", np.nan),
            "durbin_watson": diag.get("durbin_watson", np.nan), "max_vif": mt.get("max_vif", np.nan),
            "significant_coef_ratio": mt.get("significant_coef_ratio", np.nan),
            "expected_signs_ok": mt.get("expected_signs_ok", None),
            "clean_score": mt.get("clean_score", np.nan),
            "selection_metric_value": _selection_value(mt, metric),
            "selected_best_model": False,
        })
    return pd.DataFrame(rows)


def select_best_model(comparison_df, results, config):
    metric = config.get("selection_metric", "bic")
    ok = comparison_df[comparison_df["status"] == "success"].copy()
    if len(ok) == 0:
        return None
    valcol = "selection_metric_value"
    sub = ok.dropna(subset=[valcol])
    if len(sub) == 0:  # metrik hesaplanamadıysa clean_score'a düş
        sub = ok.dropna(subset=["clean_score"])
        valcol, metric = "clean_score", "clean_score"
        if len(sub) == 0:
            best = ok.iloc[0]["model_name"]
            comparison_df["selected_best_model"] = comparison_df["model_name"] == best
            return best
    if metric in _HIGHER_BETTER:
        best = sub.loc[sub[valcol].idxmax(), "model_name"]
    else:
        best = sub.loc[sub[valcol].idxmin(), "model_name"]
    comparison_df["selected_best_model"] = comparison_df["model_name"] == best
    return best

## 8. Otomatik Türkçe yorum (özet rapor)

Kaç model çalıştı, en iyi model hangisi ve neden seçildi, testler temiz mi, en
büyük sorun ne, forecast ne söylüyor ve model kullanılabilir mi — hepsi kısa ve
net Türkçe cümlelerle özetlenir.

In [ ]:
def _fmt(v, nd=4):
    try:
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return "NA"
        return f"{v:.{nd}g}"
    except Exception:
        return str(v)


def generate_text_report(results, comparison_df, best_model, forecast_results, config):
    n_ok = sum(1 for r in results if r["status"] == "success")
    n_fail = sum(1 for r in results if r["status"] == "failed")
    metric = config.get("selection_metric", "bic")
    L = []
    L.append(f"Toplam {len(results)} model denendi: {n_ok} basarili, {n_fail} hatali calisti.")
    if n_fail:
        failed = [r["model_name"] for r in results if r["status"] == "failed"]
        L.append(f"Hata veren modeller: {', '.join(failed)} (ayrinti icin 10_ERRORS_WARNINGS).")

    if best_model is None:
        L.append("Hicbir model basariyla secilemedi; CONFIG ve veri kontrol edilmeli.")
        return "\n".join(L)

    best_res = next(r for r in results if r["model_name"] == best_model)
    brow = comparison_df[comparison_df["model_name"] == best_model].iloc[0]
    diag = best_res["diagnostics"]

    L.append(f"Secim kriteri '{metric}' oldugu icin en iyi model olarak "
             f"{best_model} secildi (kriter degeri={_fmt(brow['selection_metric_value'])}, "
             f"clean_score={_fmt(brow['clean_score'], 3)}/100).")

    def sent(key, ok_txt, bad_txt):
        v = diag.get(key)
        if v is None or np.isnan(v):
            return None
        return f"{ok_txt} (p={v:.3f})" if v >= 0.05 else f"{bad_txt} (p={v:.3f})"

    for key, ok_t, bad_t in [
        ("bg_pvalue", "Breusch-Godfrey: otokorelasyon bulgusu yok",
         "Breusch-Godfrey: otokorelasyon riski var"),
        ("bp_pvalue", "Breusch-Pagan: sabit varyans reddedilemiyor",
         "Breusch-Pagan: degisen varyans riski var, robust standart hatalar tercih edilmeli"),
        ("jb_pvalue", "Jarque-Bera: artiklar normal dagiliyor",
         "Jarque-Bera: artiklar normal dagilmiyor"),
        ("reset_pvalue", "RESET: spesifikasyon uygun",
         "RESET: model spesifikasyonu sorunlu olabilir"),
    ]:
        s = sent(key, ok_t, bad_t)
        if s:
            L.append(s)

    mv = brow.get("max_vif")
    if mv is not None and not (isinstance(mv, float) and np.isnan(mv)):
        L.append(interpret_vif(mv, config["clean_score"]["vif_threshold"]).capitalize() + ".")

    problems = []
    for key, lab in [("bg_pvalue", "otokorelasyon"), ("bp_pvalue", "degisen varyans (BP)"),
                     ("white_pvalue", "degisen varyans (White)"), ("reset_pvalue", "spesifikasyon")]:
        v = diag.get(key)
        if v is not None and not np.isnan(v) and v < 0.05:
            problems.append(lab)
    if mv is not None and not (isinstance(mv, float) and np.isnan(mv)) and mv > config["clean_score"]["vif_threshold"]:
        problems.append("coklu dogrusal baglanti")
    L.append("En dikkat cekici sorun(lar): " + (", ".join(problems) if problems else "belirgin sorun yok") + ".")

    fser = forecast_results.get("forecasts", {}).get(best_model)
    if fser is not None and len(fser) > 0:
        first, last = float(fser.iloc[0]), float(fser.iloc[-1])
        direction = "kademeli yukselis" if last > first * 1.001 else (
            "kademeli dusus" if last < first * 0.999 else "yatay seyir")
        L.append(f"Forecast tarafinda {best_model}, hedef degiskenin {fser.index[0].date()} - "
                 f"{fser.index[-1].date()} araliginda {first:.4g} -> {last:.4g} ({direction}) "
                 f"seyrettigini gosteriyor.")

    cs = brow.get("clean_score")
    if (cs is not None and not (isinstance(cs, float) and np.isnan(cs)) and cs >= 70) and not problems:
        L.append("Sonuc: Model forecast icin makul gorunuyor; yine de senaryo varsayimlari gozden gecirilmeli.")
    else:
        L.append("Sonuc: Model kullanilabilir ancak tani sorunlari nedeniyle forecast oncesi revizyon "
                 "(gecikme yapisi ve/veya robust standart hatalar) onerilir.")
    return "\n".join(L)

## 9. Excel çıktısı

Tüm sonuçlar tek bir Excel dosyasına (11 sayfa) yazılır. Sayfalar: `00_README`,
`01_RAW_DATA`, `02_TRANSFORMED_DATA`, `03_MODEL_COMPARISON`,
`04_BEST_MODEL_SUMMARY`, `05_COEFFICIENTS_ALL`, `06_DIAGNOSTICS_ALL`,
`07_VIF_ALL`, `08_FORECASTS`, `09_SCENARIOS`, `10_ERRORS_WARNINGS`.
Biçim: kalın başlıklar, freeze panes, autofilter, otomatik kolon genişliği,
p-value kolonları 4 ondalık, büyük sayılar binlik ayraçlı, tarih formatı düzgün.

In [ ]:
def _write_sheet(writer, df, sheet, pval_cols=None, date_cols=None, max_width=48):
    pval_cols = pval_cols or []
    date_cols = date_cols or []
    df = df.copy()
    df.to_excel(writer, sheet_name=sheet, index=False, startrow=0)
    wb, ws = writer.book, writer.sheets[sheet]
    header_fmt = wb.add_format({"bold": True, "bg_color": "#1F4E78", "font_color": "white",
                                "border": 1, "align": "center", "valign": "vcenter"})
    num_fmt = wb.add_format({"num_format": "#,##0.00"})
    int_fmt = wb.add_format({"num_format": "#,##0"})
    p_fmt = wb.add_format({"num_format": "0.0000"})
    nrows, ncols = df.shape
    for c, col in enumerate(df.columns):
        ws.write(0, c, str(col), header_fmt)
        series = df[col]
        try:
            maxlen = int(series.astype(str).map(len).max())
        except Exception:
            maxlen = 10
        width = min(max_width, max(len(str(col)), maxlen) + 2)
        if col in date_cols:
            ws.set_column(c, c, max(12, width))
        elif col in pval_cols:
            ws.set_column(c, c, width, p_fmt)
        elif pd.api.types.is_integer_dtype(series):
            ws.set_column(c, c, width, int_fmt)
        elif pd.api.types.is_float_dtype(series):
            ws.set_column(c, c, width, num_fmt)
        else:
            ws.set_column(c, c, width)
    if ncols > 0:
        ws.autofilter(0, 0, max(nrows, 1), ncols - 1)
    ws.freeze_panes(1, 0)


def write_readme(writer, prepared_df, config, best_model, report):
    wb = writer.book
    ws = wb.add_worksheet("00_README")
    writer.sheets["00_README"] = ws
    title_fmt = wb.add_format({"bold": True, "font_size": 16, "font_color": "#1F4E78"})
    key_fmt = wb.add_format({"bold": True, "bg_color": "#DDEBF7", "border": 1, "valign": "top"})
    val_fmt = wb.add_format({"border": 1, "valign": "top", "text_wrap": True})
    ws.set_column(0, 0, 26)
    ws.set_column(1, 1, 95)
    ws.write(0, 0, "Ekonometrik Modelleme Ciktilari", title_fmt)
    rows = [
        ("Notebook", "Konfigurasyon odakli, yeniden kullanilabilir ekonometrik modelleme"),
        ("Girdi dosyasi", str(config.get("input_file"))),
        ("Tarih araligi", f"{prepared_df.index.min().date()} - {prepared_df.index.max().date()}"),
        ("Frekans", str(config.get("frequency"))),
        ("Hedef degisken", str(config.get("target_col"))),
        ("Hedef transform", str(config.get("target_transform"))),
        ("Calistirilan modeller", ", ".join(config.get("models_to_run", []))),
        ("Secim kriteri", str(config.get("selection_metric"))),
        ("En iyi model", str(best_model)),
        ("Aktif senaryo", str(config.get("active_scenario"))),
        ("Test boyutu", str(config.get("test_size"))),
        ("Uretim zamani", pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")),
    ]
    r = 2
    for k, v in rows:
        ws.write(r, 0, k, key_fmt)
        ws.write(r, 1, v, val_fmt)
        r += 1
    r += 1
    ws.write(r, 0, "Genel Yorum", key_fmt)
    nlines = max(6, report.count("\n") + 1)
    ws.merge_range(r, 1, r + nlines, 1, report, val_fmt)
    ws.freeze_panes(2, 0)


def write_best_summary(writer, results, comparison_df, best_model, config):
    sheet = "04_BEST_MODEL_SUMMARY"
    wb = writer.book
    if best_model is None:
        pd.DataFrame({"Bilgi": ["En iyi model secilemedi."]}).to_excel(writer, sheet_name=sheet, index=False)
        return
    best_res = next(r for r in results if r["model_name"] == best_model)
    brow = comparison_df[comparison_df["model_name"] == best_model].iloc[0]
    stats = brow.to_frame("deger")
    stats.index.name = "metrik"
    stats = stats.reset_index()
    coef = best_res["coefficients"].copy()
    diag = best_res["diagnostics"]
    diag_df = pd.DataFrame([{"test": k, "deger": diag.get(k)} for k in diag])
    comments = best_res["comments"]
    com_df = pd.DataFrame({"konu": list(comments.keys()),
                           "yorum": [str(v) for v in comments.values()]})
    title_fmt = wb.add_format({"bold": True, "font_size": 13, "font_color": "#1F4E78"})
    hdr_fmt = wb.add_format({"bold": True, "bg_color": "#1F4E78", "font_color": "white", "border": 1})

    def block(title, df, startrow):
        df.to_excel(writer, sheet_name=sheet, index=False, startrow=startrow + 1)
        ws = writer.sheets[sheet]
        ws.write(startrow, 0, title, title_fmt)
        for c, col in enumerate(df.columns):
            ws.write(startrow + 1, c, str(col), hdr_fmt)
        return startrow + 1 + len(df) + 3

    r = block(f"EN IYI MODEL: {best_model} - Ana Istatistikler", stats, 0)
    r = block("Katsayi Tablosu", coef, r)
    r = block("Tani Testleri", diag_df, r)
    r = block("Turkce Yorum", com_df, r)
    ws = writer.sheets[sheet]
    ws.set_column(0, 0, 30)
    ws.set_column(1, 3, 22)
    ws.set_column(4, 10, 16)
    ws.freeze_panes(1, 0)


def build_all_coefficients(results):
    cols = ["model_name", "variable", "coef", "std_error", "t_stat", "p_value",
            "conf_low", "conf_high", "significance_comment", "expected_sign", "sign_ok"]
    frames = []
    for res in results:
        c = res.get("coefficients")
        if res["status"] != "success" or c is None or len(c) == 0:
            continue
        c = c.copy()
        c.insert(0, "model_name", res["model_name"])
        for col in cols:
            if col not in c.columns:
                c[col] = np.nan
        frames.append(c[cols])
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=cols)


def build_all_diagnostics(results):
    keys = ["bg_pvalue", "ljungbox_pvalue", "bp_pvalue", "white_pvalue",
            "jb_pvalue", "reset_pvalue", "durbin_watson"]
    rows = []
    for res in results:
        d = res.get("diagnostics", {}) or {}
        dc = interpret_diagnostics(d) if d else {}
        row = {"model_name": res["model_name"], "status": res["status"]}
        row.update({k: d.get(k, np.nan) for k in keys})
        row.update({"bg_yorum": dc.get("bg", ""), "bp_yorum": dc.get("bp", ""),
                    "jb_yorum": dc.get("jb", ""), "reset_yorum": dc.get("reset", ""),
                    "dw_yorum": dc.get("dw", "")})
        rows.append(row)
    return pd.DataFrame(rows)


def build_all_vif(results):
    frames = []
    for res in results:
        v = res.get("vif")
        if v is not None and len(v) > 0:
            v = v.copy()
            v.insert(0, "model_name", res["model_name"])
            frames.append(v)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["model_name", "variable", "VIF"])


def build_forecast_table(forecast_results, best_model):
    fi = forecast_results.get("future_index")
    if fi is None or len(fi) == 0:
        return pd.DataFrame(columns=["date"])
    df = pd.DataFrame({"date": pd.to_datetime(fi)})
    for name, ser in forecast_results.get("forecasts", {}).items():
        df[name] = pd.Series(ser).reindex(fi).values
    if best_model and best_model in forecast_results.get("forecasts", {}):
        df[f"SELECTED_{best_model}"] = pd.Series(forecast_results["forecasts"][best_model]).reindex(fi).values
    return df


def build_scenarios_table(config):
    rows = []
    for scen_name, vars_ in config.get("forecast_scenarios", {}).items():
        for var, sp in vars_.items():
            params = {k: v for k, v in sp.items() if k != "type"}
            rows.append({"scenario": scen_name, "active": scen_name == config.get("active_scenario"),
                         "variable": var, "type": sp.get("type"), "parameters": str(params)})
    if not rows:
        rows.append({"scenario": "-", "active": False, "variable": "-", "type": "-", "parameters": "-"})
    return pd.DataFrame(rows)


def build_errors_table(results, prepared_df, forecast_results):
    rows = []
    for n in STARTUP_NOTES:
        rows.append({"type": "startup", "source": "paketler", "message": n})
    meta = prepared_df.attrs.get("meta", {})
    for w in meta.get("warnings", []):
        rows.append({"type": "veri", "source": "prepare_data", "message": w})
    mr = meta.get("missing_report")
    if mr is not None:
        for col, cnt in mr.items():
            if cnt and cnt > 0:
                rows.append({"type": "eksik_deger", "source": str(col), "message": f"{int(cnt)} eksik deger dolduruldu/islenrdi"})
    for w in forecast_results.get("scenario_warnings", []):
        rows.append({"type": "forecast", "source": "senaryo", "message": w})
    for name, err in forecast_results.get("errors", {}).items():
        rows.append({"type": "forecast_hata", "source": name, "message": err})
    for res in results:
        if res["status"] == "failed":
            rows.append({"type": "model_hata", "source": res["model_name"], "message": res.get("error", "")})
    if not rows:
        rows.append({"type": "info", "source": "-", "message": "Uyari veya hata yok."})
    return pd.DataFrame(rows)


def export_to_excel(raw_df, prepared_df, results, comparison_df, best_model,
                    forecast_results, report, config, rolling_results=None):
    path = config.get("output_excel", "model_outputs.xlsx")
    dcol = config.get("date_col", "date")
    with pd.ExcelWriter(path, engine="xlsxwriter", datetime_format="yyyy-mm-dd",
                        date_format="yyyy-mm-dd") as writer:
        write_readme(writer, prepared_df, config, best_model, report)

        raw_out = raw_df.copy()
        _write_sheet(writer, raw_out, "01_RAW_DATA",
                     date_cols=[dcol] if dcol in raw_out.columns else [])

        tdf = prepared_df.reset_index()
        tdf = tdf.rename(columns={tdf.columns[0]: dcol})
        _write_sheet(writer, tdf, "02_TRANSFORMED_DATA", date_cols=[dcol])

        _write_sheet(writer, comparison_df, "03_MODEL_COMPARISON",
                     pval_cols=[c for c in comparison_df.columns if "pvalue" in c])

        write_best_summary(writer, results, comparison_df, best_model, config)

        _write_sheet(writer, build_all_coefficients(results), "05_COEFFICIENTS_ALL",
                     pval_cols=["p_value"])
        _write_sheet(writer, build_all_diagnostics(results), "06_DIAGNOSTICS_ALL",
                     pval_cols=["bg_pvalue", "ljungbox_pvalue", "bp_pvalue", "white_pvalue",
                                "jb_pvalue", "reset_pvalue"])
        _write_sheet(writer, build_all_vif(results), "07_VIF_ALL")
        _write_sheet(writer, build_forecast_table(forecast_results, best_model),
                     "08_FORECASTS", date_cols=["date"])
        _write_sheet(writer, build_scenarios_table(config), "09_SCENARIOS")
        _write_sheet(writer, build_errors_table(results, prepared_df, forecast_results),
                     "10_ERRORS_WARNINGS")

        if rolling_results is not None and len(rolling_results.get("tidy", [])) > 0:
            _write_sheet(writer, rolling_results["tidy"], "11_ROLLING_OLS",
                         pval_cols=["p_value"], date_cols=["date"])
    return path

## 9.1 Kayan pencere (rolling) OLS — zamanla değişen katsayılar

`CONFIG["rolling"]` ile yönetilen **opsiyonel** analiz. Seçilen regresyon
spesifikasyonu (varsayılan `OLS`) sabit genişlikte bir pencere ile kaydırılarak
her dönem yeniden tahmin edilir; katsayıların ve standart hataların **zaman
içindeki seyri** çıkarılır. Ana model seçim/forecast akışını **etkilemez**;
`enabled=False` iken hiç çalışmaz. Çıktı: `11_ROLLING_OLS` Excel sayfası ve
±2 standart hata bantlı katsayı grafiği.

In [ ]:
def run_rolling_ols(prepared_df, config):
    """CONFIG['rolling'] ayarına göre kayan pencere OLS katsayılarını üretir."""
    rc = config.get("rolling", {})
    if not rc.get("enabled", False):
        return None
    if not HAVE.get("rolling", False):
        return {"error": "statsmodels RollingOLS bulunamadi; kayan pencere atlandi.",
                "tidy": pd.DataFrame()}
    model_name = rc.get("model", "OLS")
    spec = config["model_specs"].get(model_name, {})
    window = int(rc.get("window", 36))
    min_nobs = int(rc.get("min_nobs", window))
    step = max(1, int(rc.get("step", 1)))
    try:
        bm = build_regression_matrix(prepared_df, spec, config)
        y, X = align_xy(bm["y"], bm["X"])
        if bm["add_constant"]:
            X = sm.add_constant(X, has_constant="add")
        if X.shape[1] == 0:
            return {"error": "Kayan pencere icin regresor bulunamadi.", "tidy": pd.DataFrame()}
        if len(y) < window:
            return {"error": f"Gozlem sayisi ({len(y)}) pencere boyundan ({window}) kucuk.",
                    "tidy": pd.DataFrame()}
        rres = RollingOLS(y, X, window=window, min_nobs=min_nobs, expanding=False).fit()
        # statsmodels sürümüne göre params ndarray dönebilir -> DataFrame'e sar
        cols, idx = list(X.columns), y.index
        params = pd.DataFrame(np.asarray(rres.params), index=idx, columns=cols)
        bse = pd.DataFrame(np.asarray(rres.bse), index=idx, columns=cols)
        tvals = pd.DataFrame(np.asarray(rres.tvalues), index=idx, columns=cols)
        pvals = pd.DataFrame(np.asarray(rres.pvalues), index=idx, columns=cols)
        recs = []
        for dt in params.index[::step]:
            for col in params.columns:
                c = params.loc[dt, col]
                if pd.isna(c):
                    continue
                p = float(pvals.loc[dt, col])
                recs.append({"date": dt, "variable": str(col), "coef": float(c),
                             "std_error": float(bse.loc[dt, col]),
                             "t_stat": float(tvals.loc[dt, col]), "p_value": p,
                             "significance_comment": interpret_pvalue(p)})
        return {"model": model_name, "window": window, "min_nobs": min_nobs, "step": step,
                "params": params, "bse": bse, "tidy": pd.DataFrame(recs), "error": None}
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}", "tidy": pd.DataFrame()}


def plot_rolling_coefficients(rolling_results, config):
    """Kayan pencere katsayılarını ±2 standart hata bandıyla çizer/kaydeder."""
    if not rolling_results or rolling_results.get("params") is None:
        return None
    params, bse = rolling_results["params"], rolling_results["bse"]
    outdir = Path(config.get("plot_dir", "plots"))
    outdir.mkdir(parents=True, exist_ok=True)
    cols = [c for c in params.columns if c != "const"] or list(params.columns)
    n = len(cols)
    fig, axes = plt.subplots(n, 1, figsize=(10, 2.6 * n), sharex=True)
    if n == 1:
        axes = [axes]
    for ax, col in zip(axes, cols):
        p = params[col].dropna()
        se = bse[col].reindex(p.index)
        ax.plot(p.index, p.values, color="#1F4E78")
        ax.fill_between(p.index, p.values - 2 * se.values, p.values + 2 * se.values,
                        color="#1F4E78", alpha=0.15)
        ax.axhline(0, color="red", lw=0.8, ls="--")
        ax.set_title(f"{col} - kayan pencere OLS (W={rolling_results['window']}) +/-2 s.h.")
        ax.grid(alpha=0.3)
    fig.tight_layout()
    path = outdir / "07_rolling_ols_coefficients.png"
    fig.savefig(path, dpi=110, bbox_inches="tight")
    try:
        plt.show()
    except Exception:
        pass
    plt.close(fig)
    return str(path)

## 10. Grafikler

Hedef değişken zaman serisi, en iyi modelin gerçek vs fitted / residual /
residual histogram grafikleri, actual vs forecast ve model karşılaştırma bar
grafiği. Grafikler `CONFIG["plot_dir"]` altına PNG olarak da kaydedilir.

In [ ]:
def make_plots(results, prepared_df, comparison_df, best_model, forecast_results, config):
    outdir = Path(config.get("plot_dir", "plots"))
    outdir.mkdir(parents=True, exist_ok=True)
    meta = prepared_df.attrs["meta"]
    tb = meta["target_base"]
    paths = []

    def _save(fig, name):
        p = outdir / name
        fig.savefig(p, dpi=110, bbox_inches="tight")
        paths.append(str(p))
        try:
            plt.show()
        except Exception:
            pass
        plt.close(fig)

    # 1) Hedef değişken zaman serisi
    try:
        fig, ax = plt.subplots(figsize=(10, 4))
        prepared_df[tb].plot(ax=ax, color="#1F4E78")
        ax.set_title(f"Hedef degisken zaman serisi: {tb}")
        ax.grid(alpha=0.3)
        _save(fig, "01_target_series.png")
    except Exception as e:
        print("Grafik 1 atlandi:", e)

    best_res = next((r for r in results if r["model_name"] == best_model), None)

    # 2) Gerçek vs fitted + 3) residual + 4) histogram (en iyi model, modellenmiş uzay)
    fitted = None
    if best_res is not None:
        try:
            fitted = pd.Series(best_res["fit_object"].fittedvalues)
        except Exception:
            fitted = None
    if fitted is not None and len(fitted) > 0:
        try:
            tm = meta["target_modeled"]
            actual = prepared_df[tm].reindex(fitted.index)
            fig, ax = plt.subplots(figsize=(10, 4))
            actual.plot(ax=ax, label="gercek", color="#1F4E78")
            fitted.plot(ax=ax, label="fitted", color="#E1701A", alpha=0.8)
            ax.set_title(f"Gercek vs Fitted (modellenmis): {best_model}")
            ax.legend(); ax.grid(alpha=0.3)
            _save(fig, "02_actual_vs_fitted.png")

            resid = (actual - fitted).dropna()
            fig, ax = plt.subplots(figsize=(10, 3.2))
            ax.plot(resid.index, resid.values, color="#555")
            ax.axhline(0, color="red", lw=1)
            ax.set_title(f"Residual plot: {best_model}")
            ax.grid(alpha=0.3)
            _save(fig, "03_residuals.png")

            fig, ax = plt.subplots(figsize=(6, 4))
            ax.hist(resid.values, bins=20, color="#1F4E78", alpha=0.85)
            ax.set_title(f"Residual histogram: {best_model}")
            _save(fig, "04_residual_hist.png")
        except Exception as e:
            print("Grafik 2-4 atlandi:", e)

    # 5) Actual vs forecast (seviye)
    fser = forecast_results.get("forecasts", {}).get(best_model)
    if fser is not None and len(fser) > 0:
        try:
            hist_tail = prepared_df[tb].iloc[-min(36, len(prepared_df)):]
            fig, ax = plt.subplots(figsize=(10, 4))
            hist_tail.plot(ax=ax, label="gercek", color="#1F4E78")
            fser.plot(ax=ax, label=f"forecast ({best_model})", color="#C00000", marker="o", ms=3)
            ax.axvline(hist_tail.index[-1], color="gray", ls="--", alpha=0.6)
            ax.set_title("Gercek vs Forecast (secilen model)")
            ax.legend(); ax.grid(alpha=0.3)
            _save(fig, "05_actual_vs_forecast.png")
        except Exception as e:
            print("Grafik 5 atlandi:", e)

    # 6) Model karşılaştırma bar (clean_score)
    try:
        ok = comparison_df[comparison_df["status"] == "success"].dropna(subset=["clean_score"])
        if len(ok) > 0:
            fig, ax = plt.subplots(figsize=(10, 4))
            colors = ["#C00000" if m == best_model else "#1F4E78" for m in ok["model_name"]]
            ax.bar(ok["model_name"], ok["clean_score"], color=colors)
            ax.set_title("Model karsilastirma: clean_score (kirmizi = secilen)")
            ax.set_ylabel("clean_score")
            plt.xticks(rotation=45, ha="right")
            ax.grid(alpha=0.3, axis="y")
            _save(fig, "06_model_comparison.png")
    except Exception as e:
        print("Grafik 6 atlandi:", e)

    return paths

## 11. Çalıştırma akışı

Aşağıdaki hücre tüm süreci **spesifikasyondaki akışla birebir** çalıştırır:
veri okuma → hazırlık → modeller → karşılaştırma → en iyi model → forecast →
rapor → grafikler → Excel. Hata veren model tüm süreci durdurmaz; hatası
`10_ERRORS_WARNINGS` sayfasına yazılır. Notebook baştan sona tek seferde çalışır.

In [ ]:
print("=" * 70)
print("1) Veri okuma ve hazirlama")
raw_df = load_data(CONFIG)
prepared_df = prepare_data(raw_df, CONFIG)
for w in prepared_df.attrs["meta"].get("warnings", []):
    print("   UYARI:", w)

print("\n2) Modeller calistiriliyor")
results = []
for model_name in CONFIG["models_to_run"]:
    result = run_model(model_name, prepared_df, CONFIG)
    flag = "OK " if result["status"] == "success" else "HATA"
    extra = "" if result["status"] == "success" else f" | {result.get('error')}"
    print(f"   [{flag}] {model_name}{extra}")
    results.append(result)

print("\n3) Karsilastirma ve en iyi model secimi")
comparison_df = compare_models(results, CONFIG)
best_model = select_best_model(comparison_df, results, CONFIG)
print(f"   Secilen en iyi model: {best_model}")

print("\n4) Forecast uretimi")
forecast_results = run_forecasts(results, prepared_df, CONFIG)
forecast_results = attach_selected_forecast(forecast_results, best_model)
for w in forecast_results.get("scenario_warnings", []):
    print("   UYARI:", w)

print("\n5) Otomatik Turkce yorum")
report = generate_text_report(results, comparison_df, best_model, forecast_results, CONFIG)
print("-" * 70)
print(report)
print("-" * 70)

if CONFIG.get("make_plots", True):
    print("\n6) Grafikler")
    try:
        make_plots(results, prepared_df, comparison_df, best_model, forecast_results, CONFIG)
    except Exception as e:
        print("   Grafik uretiminde hata:", e)

print("\n6.1) Kayan pencere (rolling) OLS")
rolling_results = run_rolling_ols(prepared_df, CONFIG)
if rolling_results is None:
    print("   Kapali (CONFIG['rolling']['enabled']=False).")
elif rolling_results.get("error"):
    print("   Not:", rolling_results["error"])
else:
    print(f"   {rolling_results['model']} icin W={rolling_results['window']} kayan pencere "
          f"katsayilari hesaplandi ({rolling_results['tidy']['date'].nunique()} pencere).")
    if CONFIG.get("make_plots", True):
        try:
            plot_rolling_coefficients(rolling_results, CONFIG)
        except Exception as e:
            print("   Rolling grafik hatasi:", e)

print("\n7) Excel ciktisi")
output_path = export_to_excel(raw_df=raw_df, prepared_df=prepared_df, results=results,
                              comparison_df=comparison_df, best_model=best_model,
                              forecast_results=forecast_results, report=report, config=CONFIG,
                              rolling_results=rolling_results)
print(f"   Excel yazildi: {output_path}")
print("=" * 70)

## 12. Sonuç tablosunu incele

Karşılaştırma tablosunu ve seçilen modelin forecast'ini aşağıda görebilirsiniz.

In [ ]:
def _show(df):
    try:
        from IPython.display import display
        display(df)
    except Exception:
        print(df.to_string())


print("Model karsilastirma tablosu:")
_show(comparison_df[["model_name", "status", "r2", "adj_r2", "aic", "bic",
                     "rmse_test", "mae_test", "mape_test", "clean_score",
                     "selection_metric_value", "selected_best_model"]])

print(f"\nSecilen model: {best_model}")
if forecast_results.get("selected_model_forecast") is not None:
    print("Secilen modelin forecast'i:")
    _show(forecast_results["selected_model_forecast"].to_frame("forecast"))